# Tahap 2 sampai 5: Eksplorasi, Praproses, Validasi, *Baseline*, dan Model Lanjutan
**Proyek:** Peramalan PM2.5 Beijing (Muhammad Alif Qadri, D082261018)

Tujuan bagian ini sederhana: **mengenal data, menyiapkannya, menetapkan acuan (*baseline*), lalu membandingkan model lanjutan dengan acuan itu** lewat *cross-validation* pada data latih. Data uji tidak dibuka di *notebook* ini.

**Istilah yang dipakai** (supaya tidak tertukar):

| Istilah | Arti |
|---|---|
| Stasiun latih | 9 stasiun yang datanya dipakai untuk melatih model |
| Stasiun uji | 3 stasiun yang **belum pernah dilihat model** saat dilatih (Dingling, Wanliu, Shunyi) |
| Periode latih | 80% waktu awal (sampai 2016-05-12 18:00), sama untuk semua stasiun |
| Periode uji | 20% waktu akhir (mulai 2016-05-12 19:00), sama untuk semua stasiun |
| Data latih | stasiun latih pada periode latih |
| Data uji | 12 stasiun pada periode uji, dilaporkan terpisah untuk stasiun latih dan stasiun uji |
| Tidak dipakai | stasiun uji pada periode latih (hanya dilihat untuk memilih stasiun uji dan menggambarkan pergeseran) |

Aturan yang dipegang sepanjang *notebook*:
1. Data uji disisihkan **di awal** dan tidak dilihat.
2. Model memprediksi PM2.5 **24 jam ke depan**, jadi fitur hanya boleh memakai data sampai jam sekarang.
3. Tidak ada langkah yang memakai nilai dari masa depan (misalnya interpolasi).
4. Validasi berurutan waktu memakai *gap* 24 jam antara data latih dan data validasi. *Fold* dibuat dari **jam unik**, bukan dari baris gabungan stasiun.

Nomor bagian mengikuti struktur *notebook* dari dosen:

| No | Isi |
|---|---|
| 00 | Pengaturan |
| 01 | Rumusan masalah (ringkas) |
| 02 | Muat data, sisihkan data uji dan stasiun uji, lalu eksplorasi (EDA) |
| 03 | Pembagian data dan skema validasi |
| 04 | Fitur, *pipeline* praproses, *baseline*, dan pemeriksaan kesesuaian desain |
| 05 | Model lanjutan (Ridge), pembandingnya (*random forest*), model tambahan (TabPFN, 05.8 sampai 05.11), dan *tuning* (hasil *cross-validation*, data uji tidak dibuka) |

Status setiap keputusan:
* **[Panduan]** dari dosen: data berurutan waktu tidak diacak, `TimeSeriesSplit`, *naive forecast* sebagai *baseline*, data uji dibuka satu kali.
* **[Keputusan penulis]** bukan aturan dosen: horizon 24 jam, nilai ekstrem dipertahankan (14 Nov 2026), tanpa *forward fill* (15 Nov 2026, dasarnya dihitung ulang di 04.6), 9 stasiun latih dan 3 stasiun uji dengan satu batas waktu untuk semua stasiun, tiga stasiun uji = Dingling, Wanliu, Shunyi (ditetapkan, dasarnya aturan di 02.2), dan identitas stasiun tidak dipakai sebagai fitur (16 Nov 2026).
* **Belum dikonfirmasi dosen:** desain 9 stasiun latih dan 3 stasiun uji.

Catatan urutan: data uji disisihkan **di awal bagian 02**, sebelum eksplorasi, sesuai Kode 1 panduan
("data uji disisihkan sekarang, sebelum eksplorasi lebih jauh"). Bagian 03 merinci aturan pembagiannya.

## 00. Pengaturan

In [ ]:
# Sumber data:
# 1) Google Drive pemilik (folder "Dataset PM25 Beijing"), jika ada.
# 2) Folder Data/Data Mentah di repo ini (dijalankan dari akar repo atau dari folder Notebook).
# 3) Jika keduanya tidak ada (misalnya dijalankan dosen), repo GitHub publik diunduh.
import os, subprocess
REPO = "https://github.com/LippyyDev/Tugas-Machine-Learning-PM25-Beijing"

FOLDER = os.environ.get("FOLDER_DATA")
if FOLDER is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        calon = "/content/drive/MyDrive/Dataset PM25 Beijing"
        FOLDER = calon if os.path.isdir(calon) else None
    except Exception:
        pass
if FOLDER is None:
    for calon in ("Data/Data Mentah", "../Data/Data Mentah"):
        if os.path.isdir(calon):
            FOLDER = calon                                # dijalankan dari dalam repo
            break
    else:
        if not os.path.isdir("repo_pm25"):
            subprocess.run(["git", "clone", "--depth", "1", REPO, "repo_pm25"], check=True)
        FOLDER = "repo_pm25/Data/Data Mentah"

# Keluaran meniru struktur repo: Hasil/Gambar, Hasil/Tabel, Data/Data Olahan.
# Di repo, akar = folder yang memuat "Data". Di Drive, akar = folder data itu sendiri.
if os.path.basename(os.path.normpath(FOLDER)) == "Data Mentah":
    AKAR = os.path.abspath(os.path.join(FOLDER, "..", ".."))
else:
    AKAR = os.path.abspath(FOLDER)
GAMBAR = os.path.join(AKAR, "Hasil", "Gambar")
TABEL = os.path.join(AKAR, "Hasil", "Tabel")
OLAHAN = os.path.join(AKAR, "Data", "Data Olahan")
for d in (GAMBAR, TABEL, OLAHAN):
    os.makedirs(d, exist_ok=True)
print("Folder data   :", FOLDER)
print("Keluaran disimpan di Hasil/Gambar, Hasil/Tabel, dan Data/Data Olahan di bawah folder:",
      os.path.basename(AKAR))

In [ ]:
import glob, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Patch
import sklearn

SEED = 42
np.random.seed(SEED)
HORIZON = 24                     # jam ke depan yang diprediksi
KOLOM_ANGKA = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

# Warna: biru = data utama, oranye = sorotan, abu-abu = pembanding atau tidak dipakai
BIRU, ORANYE, ABU = "#2a78d6", "#eb6834", "#b4b2a9"
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10, "figure.dpi": 100})

# Nama kolom yang mudah dibaca untuk grafik
NAMA_KOLOM = {"PM2.5": "PM2.5 (partikel halus)", "PM10": "PM10 (partikel kasar)", "SO2": "SO2", "NO2": "NO2", "CO": "CO",
              "O3": "O3", "TEMP": "TEMP (suhu)", "PRES": "PRES (tekanan udara)", "DEWP": "DEWP (titik embun)",
              "RAIN": "RAIN (hujan)", "WSPM": "WSPM (kecepatan angin)", "wd": "wd (arah angin)"}

def judul(ax, utama, keterangan=None):
    # judul tebal di kiri, satu kalimat cara membaca di bawahnya
    ax.set_title(utama, loc="left", fontsize=12, fontweight="bold", pad=28 if keterangan else 10)
    if keterangan:
        ax.text(0, 1.025, keterangan, transform=ax.transAxes, fontsize=9, color="#52514e", va="bottom")

def angka(x):
    return f"{x:,.0f}".replace(",", ".")            # 35064 menjadi 35.064

def simpan_gambar(nama):
    plt.tight_layout()
    plt.savefig(os.path.join(GAMBAR, nama), dpi=150)
    plt.show()

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)

## 01. Rumusan masalah (ringkas)
* **Target:** konsentrasi PM2.5 pada **24 jam ke depan** di setiap stasiun. Satu model dilatih dari gabungan **9 stasiun latih**.
* **Fitur:** hanya data yang sudah tersedia saat prediksi dibuat (polutan, cuaca, waktu), dihitung **per stasiun**, tanpa mencampur deret antar stasiun.
* **Pengujian:** pada 20% periode terakhir, dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji (stasiun yang tidak pernah dilihat model).
* **Metrik utama:** RMSE, karena galat besar saat polusi tinggi paling merugikan. MAE sebagai pendamping.

Rumusan lengkap ada di proposal.

## 02. Muat data dan eksplorasi
### 02.1 Muat data
Ada 12 berkas CSV, satu per stasiun. Semuanya digabung menjadi satu tabel.

In [ ]:
berkas = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))
df = pd.concat([pd.read_csv(f) for f in berkas], ignore_index=True)
df["waktu"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "waktu"]).reset_index(drop=True)

per_stasiun = df.groupby("station")["waktu"].agg(baris="size", jam_unik="nunique")
assert (per_stasiun["baris"] == per_stasiun["jam_unik"]).all()      # tidak ada jam ganda dalam satu stasiun
print(f"{len(berkas)} berkas, {df.shape[0]:,} baris, {df.shape[1]} kolom")
print(f"Baris per stasiun: {per_stasiun['baris'].min():,} sampai {per_stasiun['baris'].max():,}")
print("Rentang waktu:", df["waktu"].min(), "sampai", df["waktu"].max())
df.head(3)

In [ ]:
# Grafik ini hanya memakai jumlah baris dan waktu. Nilai PM2.5 belum disentuh.
urut = per_stasiun["baris"].sort_values()
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(urut.index, urut, color=BIRU)
for y, v in enumerate(urut):
    ax.text(v + urut.max() * 0.01, y, angka(v), va="center", fontsize=9)
sama = "Jumlahnya sama di semua stasiun." if urut.nunique() == 1 else "Jumlahnya tidak sama antar stasiun."
judul(ax, "Data yang dimuat: jumlah jam tercatat per stasiun",
      f"{len(urut)} stasiun, data per jam dari {df['waktu'].min():%Y-%m-%d} sampai {df['waktu'].max():%Y-%m-%d}. {sama}")
ax.set(xlabel="Jumlah jam (satu baris = satu jam)", xlim=(0, urut.max() * 1.12))
ax.xaxis.set_major_formatter(lambda x, _: angka(x))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_01_jam_per_stasiun.png")

### 02.2 Sisihkan data uji dan stasiun uji (sebelum eksplorasi)
Dua pemotongan dibuat di sini, sebelum eksplorasi:

1. **Menurut waktu:** data uji = **20% periode terakhir**, tanpa diacak. Batas waktunya **sama untuk semua stasiun** (satu stempel waktu), supaya setiap stasiun diuji pada periode yang sama dan tidak ada jam uji yang ikut terlatih lewat stasiun lain. Stasiun Beijing saling berkorelasi tinggi pada jam yang sama, jadi memotong per stasiun akan membocorkan masa depan.
2. **Menurut stasiun:** 3 dari 12 stasiun dijadikan **stasiun uji**. Model tidak pernah dilatih atau divalidasi dengan baris dari stasiun itu. Hanya 20% periode terakhirnya yang dipakai, sebagai uji "stasiun yang belum pernah dilihat" (pergeseran distribusi, tantangan wajib ketiga).

| Blok | Stasiun | Periode | Dipakai untuk |
|---|---|---|---|
| Latih | 9 stasiun latih | 80% awal | melatih dan validasi berurutan waktu |
| Uji pada stasiun latih | 9 stasiun latih | 20% akhir | uji akhir, stasiun yang dikenal model |
| Tidak dipakai | 3 stasiun uji | 80% awal | tidak melatih dan tidak memvalidasi |
| Uji pada stasiun uji | 3 stasiun uji | 20% akhir | uji akhir, stasiun yang belum pernah dilihat model |

**Tiga stasiun uji: Dingling, Wanliu, Shunyi [Keputusan penulis, 16 Nov 2026, ditetapkan sebelum data uji dibuka].** Dasarnya aturan berikut:
* Stasiun diurutkan menurut jarak distribusi PM2.5 (jarak Wasserstein) terhadap gabungan 11 stasiun lain, hanya dari periode **sebelum** batas waktu.
* Yang dijadikan stasiun uji: urutan ke-1 (paling berbeda), ke-6 (sedang), dan ke-12 (paling mirip), supaya uji mencakup beberapa tingkat pergeseran.
* Tidak ada model yang dilatih untuk memilih, dan data uji tidak dilihat.

**Batasnya:** pemilihan memakai data periode latih dari stasiun yang kemudian dijadikan stasiun uji, jadi hasil uji pada tiga stasiun itu adalah tiga contoh, bukan perkiraan umum untuk "stasiun mana pun".

Sel di bawah menghitung ulang aturan itu dan memastikan hasilnya sama dengan daftar yang ditetapkan. Mengganti daftar berarti mengubah keputusan, dan seluruh *notebook* harus dijalankan ulang dari awal.

In [ ]:
jam_unik = np.sort(df["waktu"].unique())
assert (np.diff(jam_unik) == np.timedelta64(1, "h")).all()    # jam berurutan tanpa lompatan, jadi 24 baris unik = 24 jam
BATAS = pd.Timestamp(jam_unik[int(len(jam_unik) * 0.8)])      # jam pertama periode uji, sama untuk semua stasiun

pra = df[df["waktu"] < BATAS]      # semua 12 stasiun sebelum BATAS: HANYA untuk memilih stasiun uji dan menggambarkan pergeseran
print("Batas data uji (jam pertama periode uji):", BATAS)

In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

# Langkah 1: jarak distribusi PM2.5 tiap stasiun terhadap 11 stasiun lain (hanya periode sebelum BATAS)
tab_dist = []
for s, g in pra.groupby("station"):
    v = g["PM2.5"].dropna().values
    lain = pra.loc[pra["station"] != s, "PM2.5"].dropna().values
    tab_dist.append(dict(stasiun=s, rata_rata=v.mean(), median=np.median(v), sd=v.std(), kuantil90=np.quantile(v, .9),
                         hilang_pct=100 * g["PM2.5"].isna().mean(),
                         wasserstein=wasserstein_distance(v, lain), KS=ks_2samp(v, lain).statistic))
tab_dist = pd.DataFrame(tab_dist).set_index("stasiun")
piv = pra.pivot_table(index="waktu", columns="station", values="PM2.5")
tab_dist["korelasi_dg_lainnya"] = [piv[s].corr(piv.drop(columns=s).mean(axis=1)) for s in tab_dist.index]
tab_dist = tab_dist.sort_values("wasserstein", ascending=False)
tab_dist.round(3)

In [ ]:
# Langkah 2: daftar stasiun uji yang DITETAPKAN, lalu dicek terhadap aturan
STASIUN_UJI = ["Dingling", "Wanliu", "Shunyi"]    # DITETAPKAN: keputusan penulis 16 Nov 2026
URUTAN_UJI = (1, 6, 12)                           # dasar aturan: urutan jarak Wasserstein dari terbesar (1) ke terkecil (12)

dari_aturan = [tab_dist.index[i - 1] for i in URUTAN_UJI]
assert sorted(dari_aturan) == sorted(STASIUN_UJI), (
    f"Aturan urutan {URUTAN_UJI} menghasilkan {dari_aturan}, berbeda dari daftar yang ditetapkan {STASIUN_UJI}")
STASIUN_LATIH = [s for s in sorted(df["station"].unique()) if s not in STASIUN_UJI]
assert len(STASIUN_UJI) == 3 and set(STASIUN_UJI) <= set(tab_dist.index)
assert len(STASIUN_LATIH) == 9 and not set(STASIUN_LATIH) & set(STASIUN_UJI)

tab_dist["kelompok"] = np.where(tab_dist.index.isin(STASIUN_UJI), "uji", "latih")
tab_dist = tab_dist.round(3)
tab_dist.to_csv(os.path.join(TABEL, "t2_tabel_distribusi_stasiun.csv"))
# Grafik: seberapa beda sebaran PM2.5 tiap stasiun (hanya periode sebelum BATAS)
fig, ax = plt.subplots(figsize=(9, 4.6))
nama_urut = [f"{i}. {s}" for i, s in enumerate(tab_dist.index, 1)]
warna = [ORANYE if k == "uji" else BIRU for k in tab_dist["kelompok"]]
ax.barh(nama_urut, tab_dist["wasserstein"], color=warna)
for y, v in enumerate(tab_dist["wasserstein"]):
    ax.text(v + tab_dist["wasserstein"].max() * 0.01, y, f"{v:.1f}", va="center", fontsize=9)
ax.invert_yaxis()
judul(ax, "Stasiun uji: seberapa beda sebaran PM2.5 tiap stasiun dari 11 stasiun lain",
      "Makin panjang batang, makin berbeda. Oranye = stasiun uji, diambil dari urutan ke-1, ke-6, dan ke-12.")
ax.set(xlabel="Jarak Wasserstein PM2.5 (µg/m³), hanya data sebelum batas data uji",
       xlim=(0, tab_dist["wasserstein"].max() * 1.12))
ax.legend(handles=[Patch(color=ORANYE, label="Stasiun uji (tidak dipakai melatih)"), Patch(color=BIRU, label="Stasiun latih")],
          loc="lower right", frameon=False)
ax.grid(axis="y", visible=False)
simpan_gambar("t2_03_jarak_stasiun_uji.png")
kor_pasangan = np.median(piv.corr().values[np.triu_indices(piv.shape[1], k=1)])
print("Stasiun uji (ditetapkan, sama dengan hasil aturan):", STASIUN_UJI)
print("Stasiun latih  :", STASIUN_LATIH)
print(f"Median korelasi PM2.5 antar pasangan stasiun pada jam yang sama (sebelum BATAS): {kor_pasangan:.2f}")
tab_dist

In [ ]:
latih = df[df["station"].isin(STASIUN_LATIH) & (df["waktu"] < BATAS)].copy()
blok = {
    "Latih (9 stasiun latih, 80% awal)":           len(latih),
    "Uji pada stasiun latih (9 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_LATIH) & (df["waktu"] >= BATAS)).sum(),
    "Tidak dipakai (3 stasiun uji, 80% awal)":   (df["station"].isin(STASIUN_UJI) & (df["waktu"] < BATAS)).sum(),
    "Uji pada stasiun uji (3 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_UJI) & (df["waktu"] >= BATAS)).sum(),
}
assert sum(blok.values()) == len(df)
for nama, n in blok.items():
    print(f"{nama:<46}{n:>9,} baris  {n / len(df):>5.0%}")
print("\nMulai di sini, eksplorasi hanya memakai `latih` (9 stasiun latih, sebelum BATAS).")

In [ ]:
# Diagram pembagian data. Hanya memakai waktu dan jumlah baris, nilai PM2.5 data uji tidak disentuh.
n_blok = list(blok.values())
tinggi_latih = len(STASIUN_LATIH) / 12
lebar_awal = 0.8
sel = [  # (kiri, bawah, lebar, tinggi, peran, jumlah, warna, warna teks)
    (0.0, 1 - tinggi_latih, lebar_awal, tinggi_latih, "LATIH", n_blok[0], BIRU, "white"),
    (lebar_awal, 1 - tinggi_latih, 1 - lebar_awal, tinggi_latih, "UJI", n_blok[1], ORANYE, "#0b0b0b"),
    (0.0, 0.0, lebar_awal, 1 - tinggi_latih, "TIDAK DIPAKAI", n_blok[2], "#d9d7d0", "#0b0b0b"),
    (lebar_awal, 0.0, 1 - lebar_awal, 1 - tinggi_latih, "UJI", n_blok[3], ORANYE, "#0b0b0b"),
]
fig, ax = plt.subplots(figsize=(9, 4.6))
for kiri, bawah, lebar, tinggi, peran, n, warna, teks in sel:
    ax.add_patch(Rectangle((kiri + 0.004, bawah + 0.006), lebar - 0.008, tinggi - 0.012, color=warna))
    ax.text(kiri + lebar / 2, bawah + tinggi / 2, f"{peran}\n{angka(n)} baris\n({n / len(df):.0%})",
            ha="center", va="center", fontsize=10, color=teks, fontweight="bold" if peran != "TIDAK DIPAKAI" else "normal")
ax.text(-0.01, 1 - tinggi_latih / 2, f"{len(STASIUN_LATIH)} stasiun latih", ha="right", va="center", fontsize=10)
ax.text(-0.01, (1 - tinggi_latih) / 2, f"{len(STASIUN_UJI)} stasiun uji", ha="right", va="center", fontsize=10)
ax.text(lebar_awal / 2, 1.02, f"80% waktu awal (sebelum {BATAS:%Y-%m-%d %H:%M})", ha="center", fontsize=10)
ax.text(lebar_awal + (1 - lebar_awal) / 2, 1.02, "20% waktu akhir", ha="center", fontsize=10)
ax.set(xlim=(-0.2, 1), ylim=(0, 1.08))
ax.axis("off")
ax.set_title("Pembagian data: latih, uji, dan stasiun uji", loc="left", fontsize=12, fontweight="bold", pad=28)
ax.text(-0.2, 1.1, "Luas kotak sebanding dengan jumlah baris. Hanya waktu dan jumlah baris yang dipakai, nilai PM2.5 data uji tidak dibuka.",
        transform=ax.transData, fontsize=9, color="#52514e", va="bottom")
simpan_gambar("t2_02_pembagian_data.png")

### 02.3 Statistik deskriptif (9 stasiun latih, digabung)

In [ ]:
latih[["PM2.5", "PM10", "TEMP", "DEWP", "PRES", "WSPM", "RAIN"]].describe().round(1)

### 02.4 Sebaran PM2.5
Seberapa sering PM2.5 bernilai rendah, dan seberapa sering sangat tinggi? Ini dasar keputusan soal nilai ekstrem.

In [ ]:
pm = latih["PM2.5"].dropna()
BATAS_TAMPIL = 500                       # nilai di atas ini tidak digambar supaya bentuk utamanya terlihat
fig, ax = plt.subplots()
ax.hist(pm[pm <= BATAS_TAMPIL], bins=50, color=BIRU, edgecolor="white", linewidth=0.5)
ax.axvline(pm.median(), color=ORANYE, lw=2, label=f"Median = {pm.median():.0f} (separuh jam di bawah nilai ini)")
ax.axvline(pm.mean(), color="#0b0b0b", lw=1.5, ls="--", label=f"Rata-rata = {pm.mean():.0f}")
ax.legend(loc="upper right", frameon=False)
ax.text(0.98, 0.62, f"{(pm > BATAS_TAMPIL).sum()} jam bernilai di atas {BATAS_TAMPIL}\nTidak digambar, tetapi tetap dipakai",
        transform=ax.transAxes, ha="right", fontsize=9, color="#52514e")
judul(ax, "Sebaran PM2.5 per jam (data latih)",
      "Rata-rata lebih tinggi dari median karena ada jam yang bernilai sangat tinggi.")
ax.set(xlabel="Konsentrasi PM2.5 (µg/m³)", ylabel="Jumlah jam (9 stasiun digabung)")
simpan_gambar("t2_04_sebaran_pm25.png")
print(f"Rata-rata {pm.mean():.1f} | median {pm.median():.1f} | kemiringan (skewness) {pm.skew():.2f}")
print(f"Jam bernilai di atas 500: {(pm > 500).sum()} dari {len(pm):,}")
pm.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1).to_frame("PM2.5")

### 02.5 Nilai ekstrem
Nilai sangat tinggi bisa berarti episode polusi nyata atau galat sensor.
Kita cek di 9 stasiun latih (data latih), lalu lihat apakah nilai itu muncul sendirian atau bersamaan di banyak stasiun.

In [ ]:
pm_semua = latih["PM2.5"]
print("Jumlah jam-stasiun dengan PM2.5:")
for ambang in [500, 700, 900]:
    print(f"  di atas {ambang}: {(pm_semua > ambang).sum():>4}")
print("  tepat 999   :", (pm_semua == 999).sum())

# Apakah nilai > 700 terjadi serempak? Hitung berapa stasiun yang juga > 300 pada jam itu.
jam_ekstrem = latih.loc[latih["PM2.5"] > 700, "waktu"].unique()
serempak = (latih[latih["waktu"].isin(jam_ekstrem)]
            .groupby("waktu")["PM2.5"].apply(lambda s: (s > 300).sum()))
print(f"\nPada {len(jam_ekstrem)} jam dengan PM2.5 > 700 di suatu stasiun, "
      f"median jumlah stasiun yang juga > 300: {serempak.median():.0f} dari {latih['station'].nunique()}")

### 02.6 Pola harian dan musiman
Kapan polusi paling tinggi? Pola ini menjadi dasar fitur kalender (jam dan bulan).

In [ ]:
per_jam = latih.groupby(latih["waktu"].dt.hour)["PM2.5"].mean()
nama_bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
per_bulan = latih.groupby(latih["waktu"].dt.month)["PM2.5"].mean()

print(f"Jam tertinggi: {per_jam.idxmax()}:00 ({per_jam.max():.0f}) | "
      f"terendah: {per_jam.idxmin()}:00 ({per_jam.min():.0f})")
print(f"Bulan tertinggi: {nama_bulan[per_bulan.idxmax()-1]} ({per_bulan.max():.0f}) | "
      f"terendah: {nama_bulan[per_bulan.idxmin()-1]} ({per_bulan.min():.0f})")

### 02.7 Nilai hilang
Dua hal yang dicek: **berapa banyak** yang hilang, dan **berapa lama** rentang nilai hilang berturut-turut (dihitung per stasiun, lalu digabung).
Lama rentang nilai hilang menentukan seberapa berbahaya tebakan pengisiannya (dibahas di 04.1).

In [ ]:
hilang = (latih.drop(columns=["No", "year", "month", "day", "hour", "station", "waktu"])
            .isna().mean().mul(100).sort_values())
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh([NAMA_KOLOM.get(k, k) for k in hilang.index], hilang,
        color=[ORANYE if k == "PM2.5" else BIRU for k in hilang.index])
for y, v in enumerate(hilang):
    ax.text(v + 0.05, y, f"{v:.1f}%", va="center", fontsize=9)
judul(ax, "Persentase jam yang datanya kosong, per kolom (data latih)",
      "Makin panjang batang, makin sering sensor tidak mencatat. Oranye = PM2.5, yang ingin diprediksi.")
ax.set(xlabel="Persen jam yang kosong (%)", xlim=(0, hilang.max() * 1.15))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_05_nilai_hilang_per_kolom.png")
tabel_hilang = hilang.round(2).to_frame("% hilang")
tabel_hilang.to_csv(os.path.join(TABEL, "t2_tabel_nilai_hilang.csv"))
tabel_hilang

In [ ]:
# Lama setiap rentang nilai hilang berturut-turut pada PM2.5, dihitung di dalam tiap stasiun
def lama_nilai_hilang(seri):
    kosong = seri.isna()
    kelompok = (kosong != kosong.shift()).cumsum()
    return kosong.groupby(kelompok).sum()[lambda s: s > 0]

rentang_hilang = pd.concat([lama_nilai_hilang(g.sort_values("waktu")["PM2.5"]) for _, g in latih.groupby("station")],
                  ignore_index=True)
kelas = pd.cut(rentang_hilang, [0, 1, 3, 6, 24, 10_000],
               labels=["1 jam", "2-3 jam", "4-6 jam", "7-24 jam", "lebih dari 24 jam"])
hitung = kelas.value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(hitung.index.astype(str), hitung, color=BIRU, width=0.6)
for x, y in zip(range(len(hitung)), hitung):
    ax.text(x, y, str(y), ha="center", va="bottom")
judul(ax, "Berapa lama data PM2.5 kosong sekali jalan? (data latih)",
      f"Tiap batang = jumlah kejadian kosong. {(rentang_hilang <= 3).mean():.0%} kejadian hanya berlangsung 1 sampai 3 jam.")
ax.set(xlabel="Lama kosong berturut-turut", ylabel="Jumlah kejadian (9 stasiun digabung)", ylim=(0, hitung.max() * 1.12))
simpan_gambar("t2_06_lama_nilai_hilang.png")
print(f"{(rentang_hilang <= 3).mean():.0%} kejadian kosong hanya berlangsung 1 sampai 3 jam.")
print(f"Total rentang nilai hilang: {len(rentang_hilang)} | median {rentang_hilang.median():.0f} jam | terpanjang {rentang_hilang.max()} jam")
for k in [1, 3, 6]:
    print(f"  Rentang <= {k} jam: {(rentang_hilang <= k).mean():.0%} dari seluruh rentang, "
          f"{rentang_hilang[rentang_hilang <= k].sum() / rentang_hilang.sum():.0%} dari jam hilang")
hitung.rename("jumlah kejadian").to_frame()

### 02.8 Apa yang berkaitan dengan PM2.5 besok?
Tabel ini menghitung korelasi antara nilai **sekarang** (jam *t*) dengan PM2.5 **24 jam kemudian** di stasiun yang sama (digeser per stasiun, lalu digabung).
Informasi ini yang benar-benar tersedia saat prediksi dibuat, dan menjadi dasar memilih fitur "kondisi sekarang".

In [ ]:
target_besok = latih.groupby("station")["PM2.5"].shift(-HORIZON)        # PM2.5 pada jam t+24, di stasiun yang sama
korelasi = latih[KOLOM_ANGKA].corrwith(target_besok).sort_values()

korelasi.round(3)

### 02.9 Seberapa jauh masa lalu masih berguna?
Korelasi PM2.5 dengan PM2.5 beberapa jam sebelumnya. Ini membantu memilih *lag* yang dipakai sebagai fitur.

In [ ]:
jarak = [24, 25, 26, 27, 30, 36, 48, 72, 96, 120, 144, 168]   # selisih jam antara dua nilai PM2.5 yang dibandingkan
kor_lag = [latih["PM2.5"].corr(latih.groupby("station")["PM2.5"].shift(j)) for j in jarak]

pd.Series(kor_lag, index=jarak, name="korelasi").round(3).to_frame().T

### 02.10 Perbandingan antarstasiun
Dasar untuk tantangan wajib ketiga: model diuji di stasiun yang belum pernah dilihat. Seberapa beda stasiunnya?
Tabel di bawah memakai data periode latih dari **semua 12 stasiun**, hanya untuk menggambarkan pergeseran distribusi. Model tidak melihat baris stasiun uji.

In [ ]:
ringkas = (tab_dist[["rata_rata", "median", "hilang_pct", "kelompok"]]
           .rename(columns={"hilang_pct": "persen_hilang"}).sort_values("rata_rata"))
ringkas.round(1).to_csv(os.path.join(TABEL, "t2_tabel_ringkas_stasiun.csv"))
ringkas.round(1)

## 03. Pembagian data dan skema validasi
Aturan ujian model, sesuai panduan untuk data berurutan waktu:

1. **Data uji** = 20% periode terakhir (disisihkan di 02.2), untuk 12 stasiun dengan satu batas waktu. Disimpan, dan dibuka **satu kali** di akhir proyek. Dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji.
2. **Data latih** = 80% periode awal dari 9 stasiun latih. Semua pemilihan model dan *tuning* memakai `TimeSeriesSplit(n_splits=5, gap=24)` di sini.
3. **Stasiun uji** tidak dipakai untuk melatih, memvalidasi, maupun menyetel apa pun.
4. Data **tidak diacak**. Setiap *fold* melatih dengan masa lalu dan memvalidasi dengan periode sesudahnya.
5. ***Gap* 24 jam** (`gap=24`). Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa *gap*, target baris latih terakhir jatuh di periode validasi, padahal nilai itu juga muncul sebagai fitur *lag* di data validasi. *Gap* 24 jam menutup celah ini.
6. **Setiap *fold* dibuat dari jam unik, bukan dari baris.** Sembilan stasiun punya jam yang sama. Bila baris gabungan dipotong langsung, satu jam yang sama bisa jatuh di data latih (stasiun A) dan di data validasi (stasiun B), padahal PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama (02.2). Dengan *fold* dari jam unik, semua stasiun mengikuti *fold* jam yang sama.

Sel di bawah membuat *fold* dan memeriksa aturan nomor 4, 5, dan 6.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Jam unik yang targetnya (t+24) masih di periode latih. Sama dengan jam baris yang dipakai model.
jam_cv = pd.DatetimeIndex(np.sort(latih["waktu"].unique()))
jam_cv = jam_cv[jam_cv < BATAS - pd.Timedelta(hours=HORIZON)]
cv = TimeSeriesSplit(n_splits=5, gap=24)          # jam_cv berurutan per jam, jadi gap=24 berarti 24 jam
fold_cv = [(jam_cv[a], jam_cv[b]) for a, b in cv.split(jam_cv)]

for i, (a, b) in enumerate(fold_cv, 1):
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min()       # target latih terakhir jatuh sebelum validasi dimulai
    assert len(a.intersection(b)) == 0                           # tidak ada jam yang sama di latih dan validasi
    print(f"Fold {i}: latih s.d. {a[-1]:%Y-%m-%d} | validasi {b[0]:%Y-%m-%d} s.d. {b[-1]:%Y-%m-%d} "
          f"| {len(a):,} jam latih, {len(b):,} jam validasi (x {len(STASIUN_LATIH)} stasiun)")
print("Data uji mulai", BATAS, "(tidak dipakai di fold mana pun)")

In [ ]:
# Peta fold: hanya waktu, tidak ada nilai PM2.5
fig, ax = plt.subplots(figsize=(9, 3.9))
t = mdates.date2num
for i, (a, b) in enumerate(fold_cv):
    ax.barh(i, t(a[-1]) - t(a[0]), left=t(a[0]), color=BIRU, height=0.6)
    ax.barh(i, t(b[-1]) - t(b[0]), left=t(b[0]), color=ORANYE, height=0.6)
ax.barh(len(fold_cv), t(df["waktu"].max()) - t(BATAS), left=t(BATAS), color=ABU, height=0.6)
ax.set_yticks(range(len(fold_cv) + 1), [f"Fold {i}" for i in range(1, len(fold_cv) + 1)] + ["Data uji"])
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
judul(ax, "Validasi berurutan waktu: latih selalu lebih awal dari validasi",
      "Tiap baris satu fold. Ada gap 24 jam antara latih dan validasi (terlalu sempit untuk terlihat). Data uji tidak dipakai.")
ax.set(xlabel="Waktu")
ax.legend(handles=[Patch(color=BIRU, label="Latih"), Patch(color=ORANYE, label="Validasi"),
                   Patch(color=ABU, label="Data uji (disimpan)")], loc="upper right", frameon=False, ncol=3)
ax.grid(axis="y", visible=False)
simpan_gambar("t3_01_fold_validasi.png")

### 03.1 Mengapa skema ini
| Pilihan | Alasan |
|---|---|
| Uji = 20% jam terakhir, tanpa diacak | Data berurutan waktu. Bila diacak, model memakai informasi dari masa depan. Panduan menetapkan skema ini untuk struktur data seperti ini. |
| Satu batas waktu untuk semua stasiun | PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama. Bila stasiun dipotong sendiri-sendiri, jam yang sama bisa ada di data latih satu stasiun dan di data uji stasiun lain. |
| 9 stasiun latih, 3 stasiun uji | Dua pertanyaan diuji terpisah: apakah model memprediksi baik di stasiun yang dikenal, dan apakah tetap baik di stasiun yang belum pernah dilihat (pergeseran distribusi). Periodenya sama, jadi selisih skor kedua kelompok berasal dari stasiun, bukan dari waktu. |
| `TimeSeriesSplit`, jendela latih melebar | Tiap *fold* melatih dengan semua masa lalu yang ada dan memvalidasi dengan periode sesudahnya, persis seperti pemakaian nyata (memprediksi ke depan). |
| 5 *fold* | Contoh di panduan. Tiap *fold* validasi sekitar enam setengah bulan, bergantian mencakup musim dingin dan musim hangat, jadi satu skor tidak hanya mewakili satu musim. |
| `gap=24` pada jam unik | Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa *gap*, target baris latih terakhir jatuh di periode validasi. |
| Data uji dibuka satu kali | Setiap keputusan (fitur, model, pengaturan) dibuat dari validasi data latih. Skor uji hanya dilaporkan, tidak dipakai untuk menyetel ulang. |

Risiko kebocoran dan penjagaannya:

| Risiko | Penjagaan di *notebook* |
|---|---|
| Fitur memakai nilai masa depan | Fitur hanya sampai jam *t*, tanpa interpolasi, dihitung per stasiun (04.1). Diuji dengan memotong deret di tengah dan membandingkan fitur (04.7) |
| Target latih jatuh di periode uji | Baris latih 24 jam terakhir dibuang, dicek dengan `assert` (04.2) |
| Target latih jatuh di periode validasi | `gap=24` pada jam unik, dicek dengan `assert` (03 dan 04.5) |
| Jam yang sama ada di latih dan validasi lewat stasiun lain | *Fold* dari jam unik, dicek dengan `assert` (03) |
| Stasiun uji ikut terlatih | Hanya stasiun latih yang masuk `fitur_latih`, dicek dengan `assert` (04.2 dan 04.7) |
| Praproses belajar dari data validasi atau uji | Imputer dan *scaler* di dalam *pipeline*, dilatih ulang tiap *fold* (04.3, 04.5) |
| Pemilihan stasiun uji memakai data uji | Hanya data sebelum batas waktu yang dipakai (02.2) |
| Data uji dipakai berulang | Disisihkan di 02.2, dibuka satu kali di tahap 6 (evaluasi final) lewat penjaga di 03.3 |

### 03.2 Keterbatasan skema ini
* **Lima *fold* itu sedikit.** Simpangan baku RMSE antar *fold* besar (lihat 04.5), jadi selisih kecil antar model tidak boleh dianggap nyata.
* **Periode uji hanya sekitar 9,5 bulan** (2016-05-12 sampai 2017-02-28), tanpa Maret sampai awal Mei. Skor uji belum tentu mewakili satu siklus musim penuh.
* **Hanya satu pembagian uji.** Skor uji adalah satu hasil, bukan rata-rata dari beberapa periode.
* **Tiga stasiun uji hanya tiga contoh,** dipilih dari data periode latih (02.2). Selisih skor antara kelompok 9 dan kelompok 3 stasiun bisa dipengaruhi oleh stasiun mana yang terpilih, dan tidak boleh dibaca sebagai perkiraan untuk stasiun mana pun.
* **Stasiun Beijing saling berkorelasi tinggi** pada jam yang sama (angkanya dicetak di 02.2). Sembilan stasiun pada satu jam bukan sembilan sampel yang bebas, dan stasiun uji yang berdekatan dengan stasiun latih lebih mudah diprediksi daripada kota lain yang polanya tidak mirip.
* **Identitas stasiun tidak dipakai sebagai fitur** (keputusan penulis 16 Nov 2026). Stasiun uji tidak punya kategori yang dipelajari model, jadi fitur itu tidak bisa dipakai di sana. Model hanya melihat polutan, cuaca, dan kalender.
* **Beda skor antar *fold* belum diuji penyebabnya.** Musim hanya salah satu dugaan.

In [ ]:
# Pemeriksaan skema: tiga kelompok data tidak saling bocor
assert not set(STASIUN_LATIH) & set(STASIUN_UJI)
assert latih["station"].isin(STASIUN_LATIH).all() and latih["waktu"].max() < BATAS
print("Stasiun latih  :", ", ".join(STASIUN_LATIH))
print("Stasiun uji:", ", ".join(STASIUN_UJI))

### 03.3 Penjaga data uji
Berkas fitur uji hanya boleh dibaca lewat fungsi di bawah (dipakai nanti di tahap 6). Fungsi ini **tidak dijalankan di sini**.

* Pembukaan pertama dicatat di `Hasil/Tabel/log_pembukaan_data_uji.csv`: waktu, nama berkas, *hash* berkas, *hash* konfigurasi model, dan alasan.
* Membaca ulang dengan konfigurasi model yang **sama** diizinkan, supaya *notebook* tetap bisa dijalankan ulang dari awal.
* Membuka lagi dengan konfigurasi yang **berbeda** ditolak. Itu tanda model disetel ulang setelah melihat skor uji.

**Batas penjaga ini:** hanya mencegah pembukaan ganda yang tidak disengaja. Orang yang menghapus baris log atau mengubah fungsi tetap bisa melewatinya, dan fungsi tidak bisa tahu apakah konfigurasi dipilih dari validasi atau dari skor uji. Aturan pemilihan model ada di `Laporan/Protokol Evaluasi Data Uji.md`, dan log ini baru berarti kalau di-*commit* bersama versi kode model yang dibekukan.

In [ ]:
import json, hashlib

LOG_UJI = os.path.join(TABEL, "log_pembukaan_data_uji.csv")
KOLOM_LOG = ["waktu", "berkas", "sha256_berkas", "sha256_konfigurasi", "alasan"]

def _sha(b):
    return hashlib.sha256(b).hexdigest()

def buka_data_uji(jalur, konfigurasi, alasan, log=LOG_UJI):
    """Satu-satunya pintu membaca berkas fitur uji. konfigurasi = dict hiperparameter, daftar fitur, dan praproses model final."""
    nama = os.path.basename(jalur)
    sha_konf = _sha(json.dumps(konfigurasi, sort_keys=True, default=str).encode())
    sha_berkas = _sha(open(jalur, "rb").read())
    riwayat = pd.read_csv(log) if os.path.exists(log) else pd.DataFrame(columns=KOLOM_LOG)
    lalu = riwayat[riwayat["berkas"] == nama]
    if len(lalu) == 0:
        baris = pd.DataFrame([[pd.Timestamp.now().isoformat(timespec="seconds"), nama,
                               sha_berkas, sha_konf, alasan]], columns=KOLOM_LOG)
        pd.concat([riwayat, baris], ignore_index=True).to_csv(log, index=False)
        print(f"Data uji {nama} dibuka untuk pertama kali dan dicatat.")
    elif sha_konf in set(lalu["sha256_konfigurasi"]):
        print(f"Data uji {nama} sudah pernah dibuka dengan konfigurasi yang sama: dibaca ulang untuk reproduksi.")
        if sha_berkas not in set(lalu["sha256_berkas"]):
            print("PERINGATAN: isi berkas berbeda dari saat pertama dibuka (cek versi pustaka atau apakah berkas dibuat ulang).")
    else:
        raise RuntimeError(f"{nama} sudah pernah dibuka dengan konfigurasi model yang lain. "
                           "Menyetel ulang model setelah melihat skor uji tidak diizinkan; laporkan skor pertama apa adanya.")
    return pd.read_csv(jalur, index_col=0, parse_dates=True)

print("Penjaga data uji siap. Belum ada berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else "Log pembukaan data uji sudah ada:", LOG_UJI if os.path.exists(LOG_UJI) else "")

## 04. Fitur dan *pipeline* praproses
### 04.1 Siapkan fitur
Setiap baris = satu jam *t* di satu stasiun. **Target** = PM2.5 pada jam *t+24* di stasiun yang sama.
Semua fitur hanya memakai data sampai jam *t*, dan dihitung **per stasiun** (*lag* dan *rolling mean* tidak pernah mengambil nilai dari stasiun lain).

| Kelompok | Isi |
|---|---|
| Kondisi sekarang | polutan dan cuaca pada jam *t* |
| *Lag* | PM2.5 beberapa jam sebelumnya |
| *Rolling mean* | rata-rata PM2.5 6, 24, dan 72 jam terakhir |
| Kalender | jam, hari dalam pekan, bulan (dalam bentuk sin dan cos agar jam 23 dekat dengan jam 0) |
| Angin | arah angin diubah menjadi sin dan cos |

Nama stasiun disimpan sebagai kolom `stasiun` untuk keperluan pelaporan, **bukan sebagai fitur**.

**Nilai hilang [Usulan, bukan aturan dosen]:**
* Target hilang: baris dibuang, tidak diisi tebakan.
* Fitur hilang: **tidak diisi di sini** (tanpa *forward fill*). Semua yang kosong diurus imputer median di dalam *pipeline* (04.3), yang hanya belajar dari data latih tiap *fold*.

Mengapa tanpa *forward fill*? Keputusan 15 Nov 2026 dibuat dari perbandingan pada pilot Tiantan: pengisian 0, 1, 3, 6, 12, dan 24 jam tidak memberi beda yang bisa dibedakan dari *noise* antar *seed*. Di 04.6 perbandingan itu diulang pada 9 stasiun latih. Bila hasilnya menunjukkan beda yang jelas melebihi *noise*, keputusan itu perlu ditinjau.

In [ ]:
FFILL_MAKS = 0      # batas forward fill (jam). 0 = tanpa pengisian (keputusan 15 Nov 2026)
ARAH = {a: i * 22.5 for i, a in enumerate(
    ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"])}

def buat_fitur(data_stasiun):
    # Satu stasiun saja. Semua operasi memakai shift positif atau jendela ke belakang.
    d = data_stasiun.set_index("waktu").sort_index()
    f = pd.DataFrame(index=d.index)

    # Target: PM2.5 asli 24 jam ke depan (tanpa pengisian)
    f["target"] = d["PM2.5"].shift(-HORIZON)

    # Kondisi sekarang. Bila FFILL_MAKS > 0, rentang nilai hilang yang pendek diisi nilai terakhir (hanya melihat ke belakang)
    isi = d[KOLOM_ANGKA].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d[KOLOM_ANGKA]
    for k in KOLOM_ANGKA:
        f[k] = isi[k]

    # Lag dan rolling mean PM2.5 (shift positif = masa lalu)
    for j in [1, 2, 3, 6, 12, 24, 48]:
        f[f"PM2.5_lag{j}"] = isi["PM2.5"].shift(j)
    for w in [6, 24, 72]:
        f[f"PM2.5_rata{w}j"] = isi["PM2.5"].rolling(w, min_periods=w // 2).mean()

    # Kalender dan arah angin dalam bentuk lingkaran
    wd = d["wd"].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d["wd"]
    sudut = wd.map(ARAH)
    f["angin_sin"], f["angin_cos"] = np.sin(np.radians(sudut)), np.cos(np.radians(sudut))
    f["jam_sin"], f["jam_cos"] = np.sin(2*np.pi*d.index.hour/24), np.cos(2*np.pi*d.index.hour/24)
    f["hari_sin"], f["hari_cos"] = np.sin(2*np.pi*d.index.dayofweek/7), np.cos(2*np.pi*d.index.dayofweek/7)
    f["bulan_sin"], f["bulan_cos"] = np.sin(2*np.pi*d.index.month/12), np.cos(2*np.pi*d.index.month/12)
    return f

def fitur_stasiun(data, daftar):
    # Fitur dihitung terpisah untuk setiap stasiun, lalu ditumpuk. Kolom `stasiun` bukan fitur.
    bagian = []
    for s in daftar:
        f = buat_fitur(data[data["station"] == s])
        f.insert(0, "stasiun", s)
        bagian.append(f)
    return pd.concat(bagian)

### 04.2 Potong fitur di batas data uji
Fitur dibuat dari deret lengkap tiap stasiun (karena hanya melihat ke belakang, ini aman), lalu dipotong di `BATAS` dan menurut stasiun:

* **Latih:** hanya 9 stasiun latih, hanya baris yang targetnya (jam *t+24*) jatuh sebelum `BATAS`.
* **Uji:** baris jam *t* di `BATAS` atau sesudahnya, dari 12 stasiun, diberi label kelompok (`stasiun_latih` atau `stasiun_uji`).
* **Dibuang:** baris 3 stasiun uji sebelum `BATAS` (tidak dipakai), dan baris 24 jam terakhir sebelum `BATAS` di semua stasiun.

**Celah kebocoran yang ditutup:** baris latih 24 jam terakhir punya target yang jatuh di periode uji.
Baris itu dibuang supaya tidak ada nilai uji yang ikut dilatih.

In [ ]:
fitur_semua = fitur_stasiun(df, STASIUN_LATIH + STASIUN_UJI)
waktu_target = fitur_semua.index + pd.Timedelta(hours=HORIZON)
ada_target = fitur_semua["target"].notna()

fitur_latih = fitur_semua[fitur_semua["stasiun"].isin(STASIUN_LATIH) & (waktu_target < BATAS) & ada_target]
fitur_uji = fitur_semua[(fitur_semua.index >= BATAS) & ada_target].copy()
fitur_uji.insert(1, "kelompok_uji", np.where(fitur_uji["stasiun"].isin(STASIUN_UJI), "stasiun_uji", "stasiun_latih"))
del fitur_semua

KOLOM_FITUR = [c for c in fitur_latih.columns if c not in ("stasiun", "target")]
n_stasiun_uji = (fitur_uji["kelompok_uji"] == "stasiun_uji").sum()
print(f"Fitur latih: {len(fitur_latih):,} baris, {len(KOLOM_FITUR)} fitur, {fitur_latih['stasiun'].nunique()} stasiun")
print(f"Fitur uji  : {len(fitur_uji):,} baris (disimpan, tidak dibuka): "
      f"{len(fitur_uji) - n_stasiun_uji:,} stasiun latih + {n_stasiun_uji:,} stasiun uji")
print("Target latih terakhir:", (fitur_latih.index.max() + pd.Timedelta(hours=HORIZON)),
      "| harus sebelum", BATAS)

assert fitur_latih.index.max() + pd.Timedelta(hours=HORIZON) < BATAS
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)                 # tidak ada baris stasiun uji di data latih
assert fitur_uji.index.min() >= BATAS
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_uji", "stasiun"]) == set(STASIUN_UJI)
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_latih", "stasiun"]) == set(STASIUN_LATIH)

In [ ]:
# Cek: berapa persen fitur yang masih kosong (tanpa pengisian, diurus imputer di pipeline)
sisa = fitur_latih[KOLOM_FITUR].isna().mean().mul(100)
print(f"Fitur dengan nilai kosong tersisa: {(sisa > 0).sum()} dari {len(sisa)}, "
      f"tertinggi {sisa.idxmax()} = {sisa.max():.2f}%")

### 04.3 *Pipeline* praproses
Langkah yang "belajar" dari data (mengisi kosong dengan median, menyamakan skala) ditaruh di dalam *pipeline*.
*Pipeline* ini dilatih ulang di setiap *fold* `TimeSeriesSplit`, jadi hanya belajar dari data latih *fold* itu.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Belum dilatih di sini. Di tahap 4, pipeline ini dilatih ulang di setiap fold (cv dari bagian 03).
praproses = Pipeline([("isi_kosong", SimpleImputer(strategy="median")),
                      ("skala", StandardScaler())])
praproses

### 04.4 Simpan hasil
Dua berkas CSV: `fitur_latih.csv` (9 stasiun latih, periode latih) dan `fitur_uji.csv` (12 stasiun, periode uji, dengan kolom `kelompok_uji`). Berkas uji hanya disimpan.

In [ ]:
fitur_latih.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_latih.csv"))
fitur_uji.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_uji.csv"))
for nama_folder, isi in (("Gambar", GAMBAR), ("Tabel", TABEL), ("Fitur", OLAHAN)):
    print(f"{nama_folder}:", ", ".join(sorted(os.listdir(isi))))

### 04.5 *Baseline* (hasil *cross-validation*)
*Baseline* adalah titik acuan. Tanpanya, sebuah angka RMSE tidak bermakna.

*Baseline* yang dipakai satu: ***naive forecast***, PM2.5 24 jam lagi dianggap sama dengan PM2.5 sekarang, di stasiun yang sama. Ini *baseline* wajib untuk deret waktu **[Panduan]**.

Aturan evaluasi:
* `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik data latih (03). Data uji dan stasiun uji tidak disentuh.
* Skor tiap *fold* dihitung pada baris validasi dari 9 stasiun digabung.
* *Naive forecast* butuh PM2.5 jam *t*. Baris validasi yang PM2.5 jam *t*-nya kosong dibuang, karena *naive forecast* tidak bisa memprediksi tanpa nilai itu.
* Hasil dilaporkan sebagai rata-rata ± simpangan baku antar *fold*. RMSE jadi metrik utama, MAE pendamping.

In [ ]:
X = fitur_latih[KOLOM_FITUR]
y = fitur_latih["target"]

def rmse(y_asli, y_tebak): return float(np.sqrt(np.mean((y_asli - y_tebak) ** 2)))
def mae(y_asli, y_tebak):  return float(np.mean(np.abs(y_asli - y_tebak)))

nama_model = ["Naive forecast"]
skor = {n: {"RMSE": [], "MAE": []} for n in nama_model}
dibuang = jumlah_validasi = 0
for jam_a, jam_b in fold_cv:
    # target baris latih harus jatuh sebelum periode validasi dimulai, dan tidak ada jam yang dipakai bersama
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X.index.isin(jam_a), X.index.isin(jam_b)
    ada = X[b]["PM2.5"].notna().to_numpy()           # naive butuh PM2.5 jam t
    dibuang += int((~ada).sum()); jumlah_validasi += int(b.sum())
    Xv, yv = X[b][ada], y[b][ada].to_numpy()

    tebak = {"Naive forecast": Xv["PM2.5"].to_numpy()}

    for n in nama_model:
        skor[n]["RMSE"].append(rmse(yv, tebak[n]))
        skor[n]["MAE"].append(mae(yv, tebak[n]))

tabel_baseline = pd.DataFrame({
    "RMSE rata-rata": [np.mean(skor[n]["RMSE"]) for n in nama_model],
    "RMSE sd":        [np.std(skor[n]["RMSE"], ddof=1) for n in nama_model],
    "MAE rata-rata":  [np.mean(skor[n]["MAE"]) for n in nama_model],
    "MAE sd":         [np.std(skor[n]["MAE"], ddof=1) for n in nama_model]}, index=nama_model).round(2)
tabel_baseline.to_csv(os.path.join(TABEL, "t4_tabel_baseline.csv"))
fig, ax = plt.subplots()
rm = skor["Naive forecast"]["RMSE"]
ax.bar([f"Fold {i}" for i in range(1, len(rm) + 1)], rm, color=BIRU, width=0.55)
for x, v in enumerate(rm):
    ax.text(x, v, f"{v:.0f}", ha="center", va="bottom", fontsize=9)
judul(ax, "Galat baseline naive forecast di tiap fold validasi (data latih)",
      "RMSE: makin kecil makin baik. Naive = tebakan PM2.5 besok sama dengan sekarang. Data uji tidak dipakai.")
ax.set(xlabel="Fold validasi", ylabel="RMSE (µg/m³)", ylim=(0, max(rm) * 1.15))
ax.grid(axis="x", visible=False)
simpan_gambar("t4_01_baseline_rmse_per_fold.png")
print(f"Baris validasi dibuang (PM2.5 jam t kosong): {dibuang} dari {jumlah_validasi:,}")
print("RMSE per fold:")
for n in nama_model:
    print(f"  {n:<15}", np.round(skor[n]["RMSE"], 1))
tabel_baseline

### 04.6 Dasar keputusan: batas *forward fill*
*Bagian pendukung: boleh dilewati saat membaca pertama.* Isinya hanya pembenaran satu keputusan (tanpa *forward fill*), tidak dipakai bagian lain.

Keputusan 15 Nov 2026: tanpa *forward fill*, diambil dari pilot Tiantan. Di sini dasarnya dihitung ulang pada **9 stasiun latih**, **hanya dari data latih** (data uji dan stasiun uji tidak dipakai). Fitur dibuat ulang per stasiun dengan batas pengisian 0, 1, 3, 6, 12, dan 24 jam, lalu dua model dinilai dengan *fold* jam unik yang sama dengan bagian 03.
Karena rentang nilai hilang yang pendek diisi nilai terakhir (hanya melihat ke belakang), tidak ada nilai masa depan yang masuk.

Yang dibandingkan adalah selisih antar pengaturan *dan* *noise* antar *seed* pada pengaturan yang sama. Selisih yang lebih kecil dari *noise* tidak boleh dianggap nyata.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

def fitur_dengan_ffill(ff):
    # Dihitung dari `latih` (9 stasiun latih, sebelum BATAS) saja. Target 24 jam terakhir otomatis kosong, jadi dibuang.
    global FFILL_MAKS
    simpan, FFILL_MAKS = FFILL_MAKS, ff
    try:
        return fitur_stasiun(latih, STASIUN_LATIH)
    finally:
        FFILL_MAKS = simpan

def model_uji(nama, seed=SEED):
    reg = Ridge(alpha=1.0) if nama == "Ridge" else HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=4, random_state=seed)
    return Pipeline([("isi", SimpleImputer(strategy="median")), ("skala", StandardScaler()), ("m", reg)])

def baris_fold(F, jam):
    x = F[F.index.isin(jam)].dropna(subset=["target"])
    return x[KOLOM_FITUR], x["target"]

daftar_ff = [0, 1, 3, 6, 12, 24]
skor_ff = {}
for ff in daftar_ff:
    F = fitur_dengan_ffill(ff)
    for nm in ["Ridge", "HGB"]:
        r = []
        for a, b in fold_cv:
            Xa, ya = baris_fold(F, a); Xb, yb = baris_fold(F, b)
            r.append(rmse(yb, model_uji(nm).fit(Xa, ya).predict(Xb)))
        skor_ff[(ff, nm)] = np.array(r)

baris_tabel = []
for nm in ["Ridge", "HGB"]:
    dasar = skor_ff[(3, nm)]
    for ff in daftar_ff:
        r = skor_ff[(ff, nm)]
        baris_tabel.append({"model": nm, "ffill_maks_jam": ff, "RMSE rata-rata": r.mean(), "RMSE sd antar fold": r.std(ddof=1),
                            "selisih vs 3 jam": (r - dasar).mean(), "fold lebih baik dari 3 jam": int(((r - dasar) < 0).sum())})
tabel_ff = pd.DataFrame(baris_tabel).round(3)
tabel_ff.to_csv(os.path.join(TABEL, "t2_tabel_ffill.csv"), index=False)
tabel_ff

In [ ]:
# Tingkat noise dasar: pengaturan yang sama (ffill = 3 jam), hanya seed HGB yang diganti
F3 = fitur_dengan_ffill(3)
rata_seed = []
for sd in range(6):
    r = []
    for a, b in fold_cv:
        Xa, ya = baris_fold(F3, a); Xb, yb = baris_fold(F3, b)
        r.append(rmse(yb, model_uji("HGB", seed=sd).fit(Xa, ya).predict(Xb)))
    rata_seed.append(np.mean(r))
rentang_seed = max(rata_seed) - min(rata_seed)
rentang_ff = {nm: float(np.ptp([skor_ff[(ff, nm)].mean() for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
sd_fold = {nm: float(np.mean([skor_ff[(ff, nm)].std(ddof=1) for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
print("RMSE rata-rata per seed HGB (ffill = 3 jam):", np.round(rata_seed, 2), "| rentang", round(rentang_seed, 2))
print("Rentang rata-rata RMSE antar pengaturan ffill :", {k: round(v, 2) for k, v in rentang_ff.items()})
print("Simpangan baku rata-rata antar fold           :", {k: round(v, 2) for k, v in sd_fold.items()})
print("Baca: rentang antar pengaturan yang tidak melebihi simpangan baku antar fold (dan, untuk HGB, rentang seed) "
      "tidak boleh dianggap beda nyata.")

### 04.7 Pemeriksaan kesesuaian dengan desain 9/3
Sel di bawah memeriksa desain pada data yang benar-benar dibuat di atas. Semua pemeriksaan memakai `assert`, jadi *notebook* berhenti bila ada yang dilanggar.

1. Stasiun latih dan stasiun uji tidak beririsan, jumlahnya 9 dan 3.
2. Berkas latih hanya berisi stasiun latih, dan target barisnya jatuh sebelum `BATAS`.
3. Berkas uji berisi jam sejak `BATAS`, untuk dua kelompok, dan stasiun uji **hanya** ada di berkas uji.
4. *Fold* validasi: *gap* minimal 24 jam dan tidak ada jam yang sama di latih dan validasi.
5. **Fitur tidak melihat masa depan:** fitur satu stasiun dihitung dari deret yang dipotong di tengah, lalu dibandingkan dengan fitur dari deret penuh pada jam yang sama. Keduanya harus sama persis (target dikecualikan, karena target memang melihat ke depan).
6. Status penjaga data uji.

In [ ]:
# 1
assert len(STASIUN_LATIH) == 9 and len(STASIUN_UJI) == 3 and not set(STASIUN_LATIH) & set(STASIUN_UJI)
# 2
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)
assert (fitur_latih.index + pd.Timedelta(hours=HORIZON) < BATAS).all() and fitur_latih["target"].notna().all()
# 3
assert (fitur_uji.index >= BATAS).all() and fitur_uji["target"].notna().all()
assert set(fitur_uji["kelompok_uji"]) == {"stasiun_latih", "stasiun_uji"}
assert not set(fitur_latih["stasiun"]) & set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_uji", "stasiun"])
# 4
for a, b in fold_cv:
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min() and len(a.intersection(b)) == 0
# 5
T0 = pd.Timestamp("2015-06-01 00:00")
for s in (STASIUN_LATIH[0], STASIUN_UJI[0]):
    penuh = buat_fitur(df[df["station"] == s]).loc[:T0, KOLOM_FITUR]
    potong = buat_fitur(df[(df["station"] == s) & (df["waktu"] <= T0)]).loc[:T0, KOLOM_FITUR]
    pd.testing.assert_frame_equal(penuh, potong)
    print(f"Fitur {s} sampai {T0} identik dengan atau tanpa data sesudahnya ({len(penuh):,} baris diperiksa).")
# 6
print("Penjaga data uji:", "BELUM ADA berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else f"log pembukaan ada di {LOG_UJI}")
print("Semua pemeriksaan desain 9/3 lolos.")

# Ringkasan visual. Baris 1 sampai 5 sudah lolos (kalau tidak, notebook berhenti di atas), baris 6 dicek sekarang.
uji_belum_dibuka = not os.path.exists(LOG_UJI)
hasil = [
    (True, "Stasiun latih (9) dan stasiun uji (3) tidak beririsan"),
    (True, "Berkas latih hanya berisi stasiun latih, target jatuh sebelum batas data uji"),
    (True, "Berkas uji mulai dari batas, stasiun uji hanya ada di berkas uji"),
    (True, "Fold validasi punya gap 24 jam dan tidak berbagi jam"),
    (True, "Fitur tidak melihat masa depan (sama persis dengan atau tanpa data sesudahnya)"),
    (uji_belum_dibuka, "Data uji belum dibuka (tidak ada log pembukaan)" if uji_belum_dibuka else "Data uji SUDAH dibuka (ada log pembukaan)"),
]
fig, ax = plt.subplots(figsize=(9, 3.4))
for i, (ok, teks) in enumerate(hasil):
    y = len(hasil) - i
    ax.text(0.0, y, "✓ LOLOS" if ok else "! CEK", fontsize=10, fontweight="bold", va="center", color="#1f7a3d" if ok else ORANYE)
    ax.text(0.14, y, teks, fontsize=10, va="center")
ax.set(xlim=(0, 1), ylim=(0.4, len(hasil) + 0.6))
ax.axis("off")
ax.set_title(f"Pemeriksaan desain 9/3: {sum(ok for ok, _ in hasil)} dari {len(hasil)} lolos", loc="left", fontsize=12, fontweight="bold", pad=28)
ax.text(0, len(hasil) + 0.9, "Pemeriksaan memakai assert, jadi notebook berhenti bila ada yang dilanggar.", fontsize=9, color="#52514e", va="bottom")
simpan_gambar("t4_02_pemeriksaan_desain.png")

## 05. Model lanjutan dan *tuning* (hasil *cross-validation*)
Tahap 5 memakai aturan evaluasi yang sama dengan *baseline* di 04.5, supaya model lanjutan dibandingkan dengan *naive forecast* pada *fold* dan baris yang sama:

* `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik data latih 9 stasiun (bagian 03). **Data uji dan stasiun uji tidak disentuh.** Data uji baru dibuka satu kali di tahap 6 (evaluasi final), setelah satu model final dipilih dari bagian ini.
* Setiap model dilatih ulang di tiap *fold*: imputer median dan *scaler* hanya belajar dari data latih *fold* itu.
* Skor dihitung pada baris validasi yang sama dengan *naive*, yaitu baris yang PM2.5 jam *t*-nya tidak kosong. Sel di bawah memeriksa dengan `assert` bahwa galat *naive* di sini sama persis dengan 04.5.
* Dilaporkan rata-rata ± simpangan baku antar *fold*: RMSE sebagai metrik utama, MAE sebagai pendamping.

**Model lanjutan** (keputusan penulis, 22 Nov 2026): **Ridge** dengan 29 fitur awal (F1). Ridge adalah regresi linear dengan penalti L2. Ia hanya punya satu hiperparameter (`alpha`), mudah dijelaskan, dan stabil bila fitur *lag* saling berkorelasi. Model pembanding (keputusan penulis, 23 Nov 2026): ***random forest***, dibahas di 05.3 sampai 05.6.

### 05.1 *Tuning* Ridge
Grid di bawah **ditetapkan sebelum dijalankan** (22 Nov 2026, [Usulan], disetujui penulis dengan meminta *tuning* dijalankan) dan tidak diubah setelah melihat hasil:

| Model | Parameter | Grid | Konfigurasi |
|---|---|---|---|
| Ridge | `alpha` | 0,1; 1; 10; 100; 1000 | 5 |

Dikunci: 29 fitur awal (F1), imputer median, *scaler* standar, `fit_intercept=True`. Aturan pemilihan (ditulis sebelum hasil dilihat):

1. Dasar: rata-rata RMSE validasi silang 5 *fold*, *fold* dan baris validasi sama dengan 04.5 (`assert` galat *naive*).
2. Ambil `alpha` dengan rata-rata RMSE terendah. Bila ada di tepi grid, dilaporkan sebagai keterbatasan dan grid tidak diperluas lagi.
3. Model final harus lebih baik dari *naive forecast*. **Data uji tidak dibuka.** Hasilnya kandidat, bukan model final, sampai konfigurasi dibekukan dan di-*commit*.

In [ ]:
from sklearn.linear_model import Ridge

GRID_RIDGE = [0.1, 1, 10, 100, 1000]

rmse_naive = np.array(skor["Naive forecast"]["RMSE"])      # skor naive dari 04.5
mae_naive = np.array(skor["Naive forecast"]["MAE"])
X7, y7 = fitur_latih[KOLOM_FITUR], fitur_latih["target"]
fold7 = []                       # praproses dilatih per fold hanya pada data latih fold itu, lalu dipakai ulang untuk semua nilai alpha
for k, (jam_a, jam_b) in enumerate(fold_cv):
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X7.index.isin(jam_a), X7.index.isin(jam_b)
    ada = X7[b]["PM2.5"].notna().to_numpy()
    pra = Pipeline([("isi_kosong", SimpleImputer(strategy="median")), ("skala", StandardScaler())])
    Xa, ya = pra.fit_transform(X7[a]), y7[a].to_numpy()
    Xv, yv = pra.transform(X7[b][ada]), y7[b][ada].to_numpy()
    assert abs(rmse(yv, X7[b][ada]["PM2.5"].to_numpy()) - skor["Naive forecast"]["RMSE"][k]) < 1e-9
    fold7.append((Xa, ya, Xv, yv))

per_fold7 = []
for al in GRID_RIDGE:
    nama = f"Ridge (alpha={al})"
    for k, (Xa, ya, Xv, yv) in enumerate(fold7):
        tebak = Ridge(alpha=al).fit(Xa, ya).predict(Xv)
        per_fold7.append({"model": nama, "alpha": al, "fold": k + 1, "RMSE": rmse(yv, tebak), "MAE": mae(yv, tebak)})
pf7 = pd.DataFrame(per_fold7)
pf7.round(4).to_csv(os.path.join(TABEL, "t5_tabel_tuning_ridge_per_fold.csv"), index=False)

acuan = pf7[pf7["model"] == "Ridge (alpha=1)"].sort_values("fold")["RMSE"].to_numpy()   # selisih berpasangan terhadap alpha=1
baris7 = []
for al in GRID_RIDGE:
    nama = f"Ridge (alpha={al})"
    g = pf7[pf7["model"] == nama].sort_values("fold"); d = g["RMSE"].to_numpy() - acuan
    baris7.append({"Model": nama, "alpha": al,
                   "RMSE rata-rata": g["RMSE"].mean(), "RMSE sd": g["RMSE"].std(ddof=1),
                   "MAE rata-rata": g["MAE"].mean(), "MAE sd": g["MAE"].std(ddof=1),
                   "Selisih vs Ridge alpha=1": d.mean(), "sd selisih": d.std(ddof=1),
                   "Fold lebih baik dari Ridge alpha=1": f"{int((d < 0).sum())} dari {len(d)}"})
ringkas7 = pd.DataFrame(baris7).set_index("Model")
ringkas7.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning_ridge.csv"))
print("Total konfigurasi:", len(ringkas7))
print(ringkas7.sort_values("RMSE rata-rata").round(2).to_string())

# Aturan pemilihan (ditulis sebelum hasil dilihat, lihat markdown di atas)
pilih7 = ringkas7["RMSE rata-rata"].idxmin()
tepi7 = ringkas7.loc[pilih7, "alpha"] in (min(GRID_RIDGE), max(GRID_RIDGE))
print(f"\nKandidat terpilih : {pilih7} | RMSE {ringkas7.loc[pilih7, 'RMSE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'RMSE sd']:.2f}, "
      f"MAE {ringkas7.loc[pilih7, 'MAE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'MAE sd']:.2f}")
print(f"Lebih baik dari naive ({rmse_naive.mean():.2f}): {bool(ringkas7.loc[pilih7, 'RMSE rata-rata'] < rmse_naive.mean())} | alpha di tepi grid: {bool(tepi7)}")
print("Ini kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")

# Grafik: RMSE rata-rata terhadap alpha
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(ringkas7["alpha"], ringkas7["RMSE rata-rata"], marker="o", color=BIRU, label="Ridge")
ax.axhline(rmse_naive.mean(), color=ABU, ls="--", lw=1, label="Naive forecast")
ax.set_xscale("log"); ax.set_ylim(60, 90)
judul(ax, "Tuning Ridge: RMSE validasi silang", "Rata-rata 5 fold, sumbu vertikal dimulai dari 60 (selisih antar titik jauh di bawah sd antar fold, sekitar 19)")
ax.set(xlabel="alpha (skala log)", ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8)
simpan_gambar("t5_03_tuning_ridge.png")


### 05.2 Ridge dibandingkan dengan *naive forecast*
Model adalah kandidat terpilih aturan 05.1. Skor dibandingkan dengan *naive forecast* pada *fold* dan baris yang sama.

In [ ]:
model_utama = pilih7
r_u = pf7[pf7["model"] == model_utama].sort_values("fold")
baris = [{"Model": "Naive forecast", "RMSE rata-rata": rmse_naive.mean(), "RMSE sd": rmse_naive.std(ddof=1),
          "MAE rata-rata": mae_naive.mean(), "MAE sd": mae_naive.std(ddof=1), "Perbaikan RMSE vs naive (%)": 0.0, "Fold lebih baik dari naive": "n/a"}]
r, m = r_u["RMSE"].to_numpy(), r_u["MAE"].to_numpy()
baris.append({"Model": model_utama, "RMSE rata-rata": r.mean(), "RMSE sd": r.std(ddof=1),
              "MAE rata-rata": m.mean(), "MAE sd": m.std(ddof=1),
              "Perbaikan RMSE vs naive (%)": 100 * (rmse_naive.mean() - r.mean()) / rmse_naive.mean(),
              "Fold lebih baik dari naive": f"{int((r < rmse_naive).sum())} dari {len(r)}"})
hasil_ridge = pd.DataFrame(baris).set_index("Model")
hasil_ridge.round(2).to_csv(os.path.join(TABEL, "t5_tabel_ridge.csv"))
pd.DataFrame({"Naive forecast": rmse_naive, model_utama: r}, index=[f"Fold {i}" for i in range(1, 6)]).round(2).to_csv(
    os.path.join(TABEL, "t5_tabel_ridge_per_fold.csv"))
# Grafik: RMSE tiap fold untuk naive dan Ridge
fig, ax = plt.subplots(figsize=(9, 4.2))
lebar = 0.36
for j, (nama, nilai, warna) in enumerate([("Naive forecast", rmse_naive, ABU), (model_utama, r, BIRU)]):
    ax.bar(np.arange(5) + (j - 0.5) * lebar, nilai, lebar, color=warna, label=nama)
ax.set_xticks(range(5)); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
judul(ax, "RMSE validasi silang: naive dan Ridge", "Ridge lebih rendah dari naive di kelima fold (9 stasiun latih, data uji tidak dibuka)")
ax.set(ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
simpan_gambar("t5_01_ridge_per_fold.png")
hasil_ridge.round(2)


### 05.3 *Tuning random forest* (pembanding Ridge)
Keputusan penulis (23 Nov 2026): pembanding Ridge adalah ***random forest***. Pertanyaan yang dijawab, **ditulis sebelum hasil dilihat**: apakah model nonlinear berbasis pohon memberi keuntungan dibanding model linear berregularisasi pada horizon 24 jam, dan apakah keuntungannya (bila ada) konsisten di kelima *fold*? *Random forest* **tidak harus lebih baik** dari Ridge. Bila kalah atau sama, itu menjadi bahan evaluasi di bagian 05.4 sampai 05.6, bukan alasan mengganti model.

**Perlakuan yang sama dengan Ridge** (bukan hanya mirip):

| Aspek | Ridge (05.1) | *Random forest* (05.3) |
|---|---|---|
| Fitur | 29 fitur awal (F1) | sama |
| Data dan *fold* | `fold7`: `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik, 9 stasiun latih | **objek `fold7` yang sama** |
| Praproses | imputer median dan *scaler* dilatih ulang di tiap *fold* hanya dari data latih *fold* itu | sama (objek yang sama; *scaler* tidak memengaruhi pohon, tetap dipakai supaya masukan identik) |
| Baris validasi | PM2.5 jam *t* tidak kosong (`assert` galat *naive* di 05.1) | sama |
| Metrik | RMSE utama, MAE pendamping, rata-rata ± sd antar *fold* | sama |
| Aturan pemilihan | RMSE rata-rata terendah, tepi *grid* dilaporkan, *grid* tidak diperluas | sama |
| Data uji | tidak dibuka | tidak dibuka |

*Grid* di bawah **ditetapkan sebelum dijalankan** (23 Nov 2026, [Usulan], disetujui penulis dengan meminta *tuning* dijalankan) dan tidak diubah setelah melihat hasil:

| Parameter | Nilai |
|---|---|
| `min_samples_leaf` | 10; 50; 200 |
| `max_features` | 0,33; 0,6 |
| **Total konfigurasi** | **6** |

Dikunci (tidak disetel): `n_estimators=100`, `max_depth=None` (kerumitan diatur lewat `min_samples_leaf`), `max_samples=0,5` (tiap pohon dilatih dari separuh baris, demi waktu komputasi pada 2 inti), `bootstrap=True`, `random_state=SEED` (42), 29 fitur F1.

Aturan pemilihan (ditulis sebelum hasil dilihat):
1. Dasar: rata-rata RMSE validasi silang 5 *fold*, *fold* dan baris yang sama dengan *naive* dan Ridge.
2. Ambil konfigurasi dengan rata-rata RMSE terendah. Bila `min_samples_leaf` terpilih ada di tepi *grid* (10 atau 200), dilaporkan sebagai keterbatasan dan *grid* tidak diperluas. `max_features` hanya punya dua nilai, jadi keduanya tepi dan dilaporkan sebagai nilai terpilih saja.
3. Kandidat terpilih dibandingkan dengan Ridge terpilih lewat selisih berpasangan per *fold* (rata-rata ± sd selisih, jumlah *fold* yang lebih baik). Dengan lima *fold*, tidak ada klaim "nyata lebih baik" hanya dari selisih rata-rata. **Data uji tidak dibuka.** Hasilnya kandidat, bukan model final.

**Catatan keterbukaan.** Pada 22 Nov 2026 pernah dijalankan eksperimen lanjutan yang juga memuat *random forest* pada data latih yang sama (sekitar 85 konfigurasi dari berbagai keluarga model, lalu dihapus dari repositori atas permintaan penulis, lihat `Laporan/Deklarasi Penggunaan AI.md` dan riwayat Git). *Grid* di atas disusun tanpa memakai hasil itu, tetapi karena percobaan lama pernah dilihat, *tuning* ini tidak sepenuhnya buta terhadap data latih, dan jumlah konfigurasi yang pernah dicoba di *fold* yang sama lebih banyak daripada 6.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import time

GRID_RF = {"min_samples_leaf": [10, 50, 200], "max_features": [0.33, 0.6]}
RF_TETAP = dict(n_estimators=100, max_depth=None, max_samples=0.5, bootstrap=True, n_jobs=2)
konfig_rf = [(l, m) for l in GRID_RF["min_samples_leaf"] for m in GRID_RF["max_features"]]

# fold7 adalah objek yang sama dengan yang dipakai Ridge di 05.1 (imputer dan scaler per fold, baris validasi sama dengan naive)
per_fold_rf, pred_rf = [], {}
for leaf, mf in konfig_rf:
    nama = f"Random forest (leaf={leaf}, mf={mf})"
    t0 = time.time()
    for k, (Xa, ya, Xv, yv) in enumerate(fold7):
        tebak = RandomForestRegressor(min_samples_leaf=leaf, max_features=mf, random_state=SEED, **RF_TETAP).fit(Xa, ya).predict(Xv)
        pred_rf[(nama, k)] = tebak
        per_fold_rf.append({"model": nama, "min_samples_leaf": leaf, "max_features": mf, "fold": k + 1,
                            "RMSE": rmse(yv, tebak), "MAE": mae(yv, tebak)})
    print(f"{nama}: selesai dalam {time.time() - t0:.0f} detik")
pf_rf = pd.DataFrame(per_fold_rf)
pf_rf.round(4).to_csv(os.path.join(TABEL, "t5_tabel_tuning_rf_per_fold.csv"), index=False)

baris_rf = []
for leaf, mf in konfig_rf:
    nama = f"Random forest (leaf={leaf}, mf={mf})"
    g = pf_rf[pf_rf["model"] == nama].sort_values("fold")
    baris_rf.append({"Model": nama, "min_samples_leaf": leaf, "max_features": mf,
                     "RMSE rata-rata": g["RMSE"].mean(), "RMSE sd": g["RMSE"].std(ddof=1),
                     "MAE rata-rata": g["MAE"].mean(), "MAE sd": g["MAE"].std(ddof=1)})
ringkas_rf = pd.DataFrame(baris_rf).set_index("Model")
ringkas_rf.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning_rf.csv"))
print("\nTotal konfigurasi:", len(ringkas_rf))
print(ringkas_rf.sort_values("RMSE rata-rata").round(2).to_string())

# Aturan pemilihan (ditulis sebelum hasil dilihat, lihat markdown di atas)
pilih_rf = ringkas_rf["RMSE rata-rata"].idxmin()
leaf_pilih, mf_pilih = int(ringkas_rf.loc[pilih_rf, "min_samples_leaf"]), float(ringkas_rf.loc[pilih_rf, "max_features"])
tepi_rf = leaf_pilih in (min(GRID_RF["min_samples_leaf"]), max(GRID_RF["min_samples_leaf"]))
print(f"\nKandidat terpilih : {pilih_rf} | RMSE {ringkas_rf.loc[pilih_rf, 'RMSE rata-rata']:.2f} ± {ringkas_rf.loc[pilih_rf, 'RMSE sd']:.2f}, "
      f"MAE {ringkas_rf.loc[pilih_rf, 'MAE rata-rata']:.2f} ± {ringkas_rf.loc[pilih_rf, 'MAE sd']:.2f}")
print(f"Lebih baik dari naive ({rmse_naive.mean():.2f}): {bool(ringkas_rf.loc[pilih_rf, 'RMSE rata-rata'] < rmse_naive.mean())} | min_samples_leaf di tepi grid: {bool(tepi_rf)}")
print("Ini kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")

# Grafik: RMSE rata-rata terhadap min_samples_leaf, satu garis per max_features
fig, ax = plt.subplots(figsize=(9, 3.8))
for mf, warna in zip(GRID_RF["max_features"], (BIRU, ORANYE)):
    sub = ringkas_rf[ringkas_rf["max_features"] == mf].sort_values("min_samples_leaf")
    ax.plot(sub["min_samples_leaf"], sub["RMSE rata-rata"], marker="o", color=warna, label=f"Random forest, max_features={mf}")
ax.axhline(rmse_naive.mean(), color=ABU, ls="--", lw=1, label="Naive forecast")
ax.axhline(ringkas7.loc[pilih7, "RMSE rata-rata"], color="#52514e", ls=":", lw=1.2, label=f"{pilih7}")
ax.set_xscale("log"); ax.set_xticks(GRID_RF["min_samples_leaf"]); ax.set_xticklabels(GRID_RF["min_samples_leaf"])
judul(ax, "Tuning random forest: RMSE validasi silang", "Rata-rata 5 fold (9 stasiun latih, data uji tidak dibuka). Sumbu vertikal tidak dimulai dari nol")
ax.set(xlabel="min_samples_leaf (skala log)", ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8)
simpan_gambar("t5_04_tuning_rf.png")

### 05.4 *Random forest* dibandingkan dengan Ridge dan *naive forecast*
Kandidat terpilih aturan 05.3, pada *fold* dan baris yang sama dengan Ridge dan *naive forecast*. Selisih berpasangan dihitung per *fold* (RMSE *random forest* dikurangi RMSE Ridge, negatif berarti *random forest* lebih baik).

In [ ]:
rr = pf7[pf7["model"] == model_utama].sort_values("fold")
rf_ = pf_rf[pf_rf["model"] == pilih_rf].sort_values("fold")
r_ridge, m_ridge = rr["RMSE"].to_numpy(), rr["MAE"].to_numpy()
r_rf, m_rf = rf_["RMSE"].to_numpy(), rf_["MAE"].to_numpy()
d_rf = r_rf - r_ridge                                    # selisih berpasangan per fold

def baris_model(nama, r, m):
    return {"Model": nama, "RMSE rata-rata": r.mean(), "RMSE sd": r.std(ddof=1), "MAE rata-rata": m.mean(), "MAE sd": m.std(ddof=1),
            "Perbaikan RMSE vs naive (%)": 100 * (rmse_naive.mean() - r.mean()) / rmse_naive.mean(),
            "Fold lebih baik dari naive": f"{int((r < rmse_naive).sum())} dari {len(r)}"}
tabel_tiga = pd.DataFrame([baris_model("Naive forecast", rmse_naive, mae_naive),
                           baris_model(model_utama, r_ridge, m_ridge),
                           baris_model(pilih_rf, r_rf, m_rf)]).set_index("Model")
tabel_tiga.round(2).to_csv(os.path.join(TABEL, "t5_tabel_rf_vs_ridge.csv"))
pd.DataFrame({"Naive forecast": rmse_naive, model_utama: r_ridge, pilih_rf: r_rf, "RF minus Ridge": d_rf},
             index=[f"Fold {i}" for i in range(1, 6)]).round(2).to_csv(os.path.join(TABEL, "t5_tabel_rf_vs_ridge_per_fold.csv"))

print(tabel_tiga.round(2).to_string())
print(f"\nRMSE per fold: naive {np.round(rmse_naive, 1)} | Ridge {np.round(r_ridge, 1)} | RF {np.round(r_rf, 1)}")
print(f"Selisih RF minus Ridge per fold: {np.round(d_rf, 2)}")
print(f"Selisih rata-rata {d_rf.mean():.2f} ± {d_rf.std(ddof=1):.2f} (sd selisih); RF lebih baik dari Ridge di {int((d_rf < 0).sum())} dari 5 fold")
print(f"Selisih MAE rata-rata (RF minus Ridge): {(m_rf - m_ridge).mean():.2f}")

fig, ax = plt.subplots(figsize=(9, 4.2))
lebar = 0.27
for j, (nama, nilai, warna) in enumerate([("Naive forecast", rmse_naive, ABU), (model_utama, r_ridge, BIRU), (pilih_rf, r_rf, ORANYE)]):
    ax.bar(np.arange(5) + (j - 1) * lebar, nilai, lebar, color=warna, label=nama)
ax.set_xticks(range(5)); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
judul(ax, "RMSE validasi silang: naive, Ridge, dan random forest", "Fold dan baris validasi sama untuk ketiganya (9 stasiun latih, data uji tidak dibuka)")
ax.set(ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
simpan_gambar("t5_05_ridge_vs_rf_per_fold.png")
tabel_tiga.round(2)

### 05.5 Kepekaan terhadap *seed*
*Random forest* bersifat acak (*bootstrap* dan pemilihan fitur), jadi satu *seed* belum cukup untuk menilai selisih dengan Ridge, yang deterministik. Konfigurasi terpilih di 05.3 dilatih ulang dengan tiga *seed* lain (43, 44, 45), tanpa mengubah konfigurasi. Ini pemeriksaan sesudah pemilihan, bukan *tuning* tambahan.

In [ ]:
seed_uji = [SEED, 43, 44, 45]
baris_seed = []
for sd in seed_uji:
    if sd == SEED:
        r_s = r_rf
    else:
        r_s = np.array([rmse(yv, RandomForestRegressor(min_samples_leaf=leaf_pilih, max_features=mf_pilih, random_state=sd, **RF_TETAP)
                                 .fit(Xa, ya).predict(Xv)) for (Xa, ya, Xv, yv) in fold7])
    baris_seed.append({"seed": sd, "RMSE rata-rata": r_s.mean(), "Selisih vs Ridge": (r_s - r_ridge).mean(),
                       "Fold lebih baik dari Ridge": f"{int((r_s < r_ridge).sum())} dari 5"})
tabel_seed = pd.DataFrame(baris_seed).set_index("seed")
tabel_seed.round(3).to_csv(os.path.join(TABEL, "t5_tabel_rf_seed.csv"))
print(tabel_seed.round(2).to_string())
rentang_seed = tabel_seed["RMSE rata-rata"].max() - tabel_seed["RMSE rata-rata"].min()
print(f"\nRentang RMSE rata-rata antar seed: {rentang_seed:.2f} | RMSE Ridge: {r_ridge.mean():.2f}")

### 05.6 Galat menurut rentang konsentrasi (diagnosis awal, data validasi)
Pertanyaan: apakah *random forest* dan Ridge salah di tempat yang sama? Prediksi dari seluruh *fold* validasi digabung (bukan rata-rata per *fold*), lalu dikelompokkan menurut **nilai PM2.5 sebenarnya** 24 jam kemudian. Rentang ditetapkan sebelum hasil dilihat untuk tujuan deskriptif: sampai 35, 35 sampai 75, 75 sampai 150, 150 sampai 250, dan di atas 250 µg/m³.

Cara membaca: *bias* = prediksi dikurangi nilai sebenarnya, jadi **negatif berarti prediksi terlalu rendah** (*underestimation*). Karena rentang ditentukan oleh nilai sebenarnya, prediktor mana pun cenderung terlihat terlalu rendah di rentang tinggi dan terlalu tinggi di rentang rendah (regresi ke rata-rata). Yang dibandingkan adalah **selisih antar model pada rentang yang sama**, bukan tanda bias itu sendiri. Ini analisis pada data validasi, bukan data uji, dan bukan pengganti analisis kesalahan di tahap 7.

In [ ]:
# Prediksi gabungan seluruh fold validasi untuk naive, Ridge terpilih, dan RF terpilih
alpha_pilih = float(ringkas7.loc[pilih7, "alpha"])
y_gab, p_naive, p_ridge, p_rf = [], [], [], []
for k, ((jam_a, jam_b), (Xa, ya, Xv, yv)) in enumerate(zip(fold_cv, fold7)):
    b = X7.index.isin(jam_b)
    ada = X7[b]["PM2.5"].notna().to_numpy()
    y_gab.append(yv)
    p_naive.append(X7[b][ada]["PM2.5"].to_numpy())
    p_ridge.append(Ridge(alpha=alpha_pilih).fit(Xa, ya).predict(Xv))
    p_rf.append(pred_rf[(pilih_rf, k)])
y_gab, p_naive, p_ridge, p_rf = (np.concatenate(z) for z in (y_gab, p_naive, p_ridge, p_rf))
assert abs(rmse(y_gab, p_ridge) - np.sqrt(np.mean([x ** 2 for x in r_ridge]))) < 1.0       # sanity: RMSE gabungan sebanding RMSE per fold
assert len(y_gab) == len(p_naive) == len(p_ridge) == len(p_rf)

batas_rentang = [0, 35, 75, 150, 250, np.inf]
nama_rentang = ["sampai 35", "35 sampai 75", "75 sampai 150", "150 sampai 250", "di atas 250"]
rentang = pd.cut(pd.Series(y_gab), bins=batas_rentang, labels=nama_rentang, right=True, include_lowest=True)
baris_r = []
for nama in nama_rentang:
    s = (rentang == nama).to_numpy()
    baris_r.append({"Rentang PM2.5 sebenarnya": nama, "Jumlah baris": int(s.sum()), "Rata-rata sebenarnya": y_gab[s].mean(),
                    "Bias naive": (p_naive[s] - y_gab[s]).mean(), "Bias Ridge": (p_ridge[s] - y_gab[s]).mean(), "Bias RF": (p_rf[s] - y_gab[s]).mean(),
                    "RMSE naive": rmse(y_gab[s], p_naive[s]), "RMSE Ridge": rmse(y_gab[s], p_ridge[s]), "RMSE RF": rmse(y_gab[s], p_rf[s])})
tabel_rentang = pd.DataFrame(baris_r).set_index("Rentang PM2.5 sebenarnya")
tabel_rentang.round(2).to_csv(os.path.join(TABEL, "t5_tabel_galat_rentang.csv"))
print(f"RMSE gabungan seluruh fold: naive {rmse(y_gab, p_naive):.2f} | Ridge {rmse(y_gab, p_ridge):.2f} | RF {rmse(y_gab, p_rf):.2f}")
print(tabel_rentang.round(1).to_string())
tabel_rentang.round(1)

**Pembacaan hasil 05.3 sampai 05.6** (angka dari keluaran sel di atas, CV 5 *fold* pada 9 stasiun latih, data uji tidak dibuka)

* ***Random forest* terpilih** (`min_samples_leaf=200`, `max_features=0,33`): RMSE 72,91 ± 19,40 dan MAE 54,38 ± 13,32, lebih baik dari *naive forecast* (86,82) di kelima *fold*. Ridge (`alpha=1000`): RMSE 72,83 ± 19,38 dan MAE 54,51 ± 13,00.
* **Selisih dengan Ridge tidak terbedakan.** Selisih RMSE berpasangan (RF dikurangi Ridge) rata-rata +0,08 dengan sd selisih 1,15. RF lebih baik di 3 dari 5 *fold* (selisih per *fold*: +1,21; +1,35; −0,24; −1,20; −0,73). Tiga *seed* lain memberi RMSE rata-rata 72,92 sampai 73,00, jadi rentang antar *seed* (0,09) jauh di bawah sd selisih antar *fold*. MAE RF lebih rendah 0,14. Kesimpulannya: pada horizon 24 jam, model nonlinear berbasis pohon **tidak memberi keuntungan yang bisa dibedakan** dari model linear berregularisasi. Tidak ada klaim bahwa salah satunya lebih baik.
* **Optimum RF ada di tepi *grid* pada kedua hiperparameter.** RMSE turun konsisten saat `min_samples_leaf` membesar (`max_features` 0,33: 74,32 di 10, 73,69 di 50, 72,91 di 200; `max_features` 0,6: 75,79, 75,21, 73,97), dan di setiap ukuran daun `max_features` 0,33 lebih baik daripada 0,6. Artinya RF yang lebih teregularisasi bisa saja sedikit lebih baik. Sesuai aturan 05.3, *grid* tidak diperluas. Ini keterbatasan: RF terpilih belum tentu RF terbaik untuk data ini.
* **Kedua model salah di tempat yang sama (05.6).** Di rentang sebenarnya di atas 250 µg/m³, bias Ridge dan RF sama-sama sekitar −200 dan RMSE-nya (216,7 dan 218,0) **lebih besar** dari *naive forecast* (171,6): keduanya memprediksi terlalu rendah saat episode polusi sangat tinggi. Di rentang sampai 35 µg/m³ keduanya memprediksi terlalu tinggi (bias +55,3 dan +57,7). Sebagian pola ini adalah efek pengelompokan menurut nilai sebenarnya (regresi ke rata-rata), dan *naive* terlihat lebih baik di rentang tinggi karena menyalin nilai sekarang, tetapi jauh lebih buruk di rentang rendah. Yang bisa dikatakan dari data ini: ketidaklinearan yang bisa ditangkap RF tidak memperbaiki *underestimation* episode tinggi. Pertanyaan etika nomor 2 (peringatan yang terlalu rendah saat polusi tinggi) tetap berlaku untuk kedua model.
* **Dugaan penyebab (belum diuji, bukan hasil):** pada horizon 24 jam, sebagian besar variasi PM2.5 mungkin ditentukan oleh cuaca esok hari yang tidak ada di fitur, sehingga tambahan kerumitan model tidak membantu. Menguji dugaan ini butuh analisis terpisah.
* **Biaya:** satu konfigurasi RF butuh 45 sampai 132 detik untuk 5 *fold* (2 inti), sedangkan Ridge hanya hitungan detik. Dengan skor yang tidak terbedakan, Ridge lebih murah dan lebih mudah dijelaskan.
* **Yang tidak boleh dilakukan dari sini:** memilih model final berdasarkan selisih 0,08 itu, atau menyetel ulang RF setelah melihat skor uji. Pemilihan model final tetap lewat aturan yang dibekukan sebelum data uji dibuka (tahap 6).

### 05.7 Cara membaca hasil dan keterbatasannya
* **Selisih kecil bukan bukti.** Selisih RMSE antar nilai `alpha` sangat kecil dibanding simpangan baku antar *fold* (sekitar 19), jadi tidak ada konfigurasi yang boleh disebut nyata lebih baik. Yang berarti adalah bentuk kurva: apakah RMSE datar di sekitar nilai terpilih atau naik tajam di salah satu sisi.
* **Skor sedikit optimistis.** Hiperparameter dipilih pada *fold* yang sama dengan yang dilaporkan, dan *naive* tidak ditala. `alpha` dipilih dari 5 nilai pada *fold* yang sama dengan yang dilaporkan. Angka ini bukan kinerja akhir.
* **Lima *fold* itu sedikit.** Simpangan baku RMSE antar *fold* besar dan sebagian besar berasal dari beda kesulitan antar periode, bukan dari beda antar model.
* **Tepi grid.** Grid tidak diperluas setelah melihat hasil. Bila optimum ada di tepi grid, itu dilaporkan, bukan dikejar.
* **Skor hanya untuk baris dengan PM2.5 jam *t* tidak kosong** (sekitar 2% baris validasi dibuang, mengikuti *naive*).
* **Identitas stasiun bukan fitur** (keputusan 16 Nov 2026), jadi hasil ini tidak menjawab apakah fitur itu membantu di stasiun latih.
* **Dua model, bukan dua kandidat yang adil sempurna.** Ridge hanya punya satu hiperparameter (5 nilai), *random forest* dua yang dicari (6 konfigurasi). Model yang lebih fleksibel dan lebih banyak konfigurasi punya peluang lebih besar mendapat skor optimistis dari pemilihan pada *fold* yang sama.
* **TabPFN (05.8 sampai 05.11) memakai konteks hasil subsampel, bukan seluruh data latih *fold*.** Model ini dirancang untuk konteks sampai puluhan ribu baris, sedangkan data latih satu *fold* berkisar 42 ribu sampai 210 ribu baris. Jadi TabPFN dan Ridge atau *random forest* tidak melihat jumlah data latih yang sama, dan selisihnya tidak boleh dibaca sebagai selisih keluarga model saja.
* **Belum dikerjakan:** pembekuan konfigurasi sebelum data uji dibuka dan evaluasi data uji (tahap 6).

### 05.8 *Tuning* TabPFN (model tambahan, perlakuan sama dengan Ridge dan *random forest*)
Pertanyaan yang dijawab, **ditulis sebelum hasil dilihat**: apakah model fondasi tabular (TabPFN, regresi *in-context* tanpa pelatihan ulang bobot) memberi keuntungan yang bisa dibedakan dari Ridge dan *random forest* pada horizon 24 jam, dan apakah keuntungannya (bila ada) konsisten di kelima *fold*? TabPFN **tidak harus lebih baik**. Bila sama atau kalah, itu dilaporkan apa adanya.

**Perlakuan yang sama** (bukan hanya mirip):

| Aspek | Ridge dan *random forest* (05.1, 05.3) | TabPFN (05.8) |
|---|---|---|
| Fitur | 29 fitur awal (F1) | sama |
| Data dan *fold* | objek `fold7`: `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik, 9 stasiun latih | **objek `fold7` yang sama** |
| Praproses | imputer median dan *scaler* dilatih ulang tiap *fold* hanya dari data latih *fold* itu | sama (masukan identik) |
| Baris validasi | PM2.5 jam *t* tidak kosong | sama (diperiksa dengan `assert` terhadap skor 05.2 dan 05.4) |
| Metrik | RMSE utama, MAE pendamping, rata-rata ± sd antar *fold* | sama; keluaran yang dipakai adalah **rata-rata** sebaran prediksi (`output_type="mean"`), karena RMSE meminimalkan galat kuadrat |
| Aturan pemilihan | RMSE rata-rata terendah, tepi *grid* dilaporkan, *grid* tidak diperluas | sama |
| Data uji | tidak dibuka | tidak dibuka |

**Perbedaan yang tidak bisa dihindari (dicatat, bukan disembunyikan):**
1. **Ukuran konteks.** TabPFN menerima sebagian kecil data latih sebagai konteks: batas rancangannya sekitar 10 ribu baris untuk versi v2 dan 50 ribu untuk versi yang lebih baru (dokumentasi paket `tabpfn`), sedangkan data latih satu *fold* 42 ribu sampai 210 ribu baris. Konteks diambil dengan **sampel acak seragam dari data latih *fold* itu saja** (`RandomState(SEED + nomor fold)`), jadi tidak ada baris validasi atau masa depan di dalamnya. Ukuran konteks ikut di-*tuning* (grid di bawah).
2. **Ia tidak "dilatih".** `fit` hanya menyimpan konteks. Praproses internal TabPFN (transformasi kuantil, dan seterusnya) juga hanya belajar dari konteks.
3. **Bobot dilatih orang lain.** TabPFN sudah dilatih awal oleh pembuatnya. Apakah data pelatihan awal itu pernah memuat deret waktu kualitas udara **belum diverifikasi** dan tidak diasumsikan. Karena itu kebocoran dari bobot awal tidak bisa dikesampingkan sepenuhnya, dan klaim apa pun harus diberi catatan ini.

*Grid* di bawah **ditetapkan sebelum dijalankan** (25 Nov 2026, [Usulan], disetujui penulis dengan meminta TabPFN dicoba) dan tidak diubah setelah melihat hasil:

| Parameter | Nilai |
|---|---|
| `n_konteks` (baris latih yang dipakai sebagai konteks) | 5.000; 10.000 |
| `n_estimators` (jumlah anggota *ensemble* TabPFN) | 4; 8 |
| **Total konfigurasi** | **4** |

Dikunci: versi model `VERSI_TABPFN = "v2"` (terbuka, tanpa token Hugging Face; versi lain butuh akun dan persetujuan lisensi), 29 fitur F1, `random_state=SEED` (42), prediksi dalam potongan 10.000 baris.

Aturan pemilihan (ditulis sebelum hasil dilihat):
1. Dasar: rata-rata RMSE validasi silang 5 *fold*, *fold* dan baris yang sama dengan *naive*, Ridge, dan *random forest*.
2. Ambil konfigurasi dengan rata-rata RMSE terendah. Bila `n_konteks` atau `n_estimators` terpilih ada di tepi *grid*, dilaporkan sebagai keterbatasan dan *grid* tidak diperluas.
3. Hasilnya kandidat, bukan model final. **Data uji tidak dibuka.**

**Pengaturan waktu jalan.** Sel di bawah paling cepat di GPU (Colab: *Runtime* > *Change runtime type* > GPU). Di CPU, memprediksi sekitar 41 ribu baris validasi per *fold* untuk 4 konfigurasi bisa memakan waktu berjam-jam. Bila terlalu lama, isi `PREDIK_MAKS_BARIS` dengan angka (misalnya 10000). Baris validasi lalu dipilih acak **sekali, sebelum ada hasil model**, sama untuk semua model, dan skor *naive*, Ridge, dan *random forest* dihitung ulang pada baris yang sama di 05.9 sehingga perbandingan tetap adil.

In [ ]:
# Pemasangan dan pengaturan TabPFN (aman dijalankan ulang)
import importlib.util, subprocess
if importlib.util.find_spec("tabpfn") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)
import torch, time
import tabpfn
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

VERSI_TABPFN = "v2"          # "v2" terbuka. Versi lain (mis. "v2.5") butuh HF_TOKEN dan persetujuan lisensi di Hugging Face
if VERSI_TABPFN != "v2" and not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata                  # Colab: simpan token di Secrets dengan nama HF_TOKEN
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
BATAS_KONTEKS = 10_000 if VERSI_TABPFN == "v2" else 50_000   # batas rancangan per versi (dokumentasi paket tabpfn)
PREDIK_MAKS_BARIS = None     # None = seluruh baris validasi (perlakuan identik dengan model lain). Angka = subset tetap, lihat markdown 05.8
PREDIK_POTONG = 10_000       # prediksi per potongan baris, hanya untuk menghemat memori

print("TabPFN", getattr(tabpfn, "__version__", "?"), "| model", VERSI_TABPFN, "| perangkat", PERANGKAT,
      "| torch", torch.__version__)
if PERANGKAT == "cpu":
    print("PERINGATAN: tanpa GPU, 4 konfigurasi x 5 fold bisa sangat lama. Pertimbangkan PREDIK_MAKS_BARIS.")

In [ ]:
GRID_TABPFN = {"n_konteks": [5_000, 10_000], "n_estimators": [4, 8]}
assert max(GRID_TABPFN["n_konteks"]) <= BATAS_KONTEKS, "grid melebihi batas konteks versi model ini"
konfig_tp = [(n, e) for n in GRID_TABPFN["n_konteks"] for e in GRID_TABPFN["n_estimators"]]

# Baris validasi yang dinilai: dipilih SEKALI, hanya dari jumlah baris (tanpa melihat prediksi atau galat model mana pun)
rng_val = np.random.RandomState(SEED)
idx_val = []
for (Xa, ya, Xv, yv) in fold7:
    if PREDIK_MAKS_BARIS is None or len(yv) <= PREDIK_MAKS_BARIS:
        idx_val.append(np.arange(len(yv)))
    else:
        idx_val.append(np.sort(rng_val.choice(len(yv), PREDIK_MAKS_BARIS, replace=False)))

def ambil_konteks(Xa, ya, n, k, seed):
    # Sampel acak seragam dari data latih fold k saja. Tidak menyentuh baris validasi.
    if len(ya) <= n:
        return Xa, ya
    pilih = np.sort(np.random.RandomState(seed + k).choice(len(ya), n, replace=False))
    return Xa[pilih], ya[pilih]

def prediksi_tabpfn(k, n_konteks, n_est, seed):
    Xa, ya, Xv, yv = fold7[k]
    Xk, yk = ambil_konteks(Xa, ya, n_konteks, k, seed)
    model = TabPFNRegressor.create_default_for_version(
        ModelVersion(VERSI_TABPFN), n_estimators=n_est, device=PERANGKAT, random_state=seed)
    model.fit(Xk, yk)
    Xs = Xv[idx_val[k]]
    tebak = np.concatenate([model.predict(Xs[i:i + PREDIK_POTONG], output_type="mean")
                            for i in range(0, len(Xs), PREDIK_POTONG)])
    assert tebak.shape == (len(Xs),) and np.isfinite(tebak).all()
    return tebak

per_fold_tp, pred_tp = [], {}
for n_konteks, n_est in konfig_tp:
    nama = f"TabPFN (konteks={n_konteks}, est={n_est})"
    t0 = time.time()
    for k in range(5):
        tebak = prediksi_tabpfn(k, n_konteks, n_est, SEED)
        yv_s = fold7[k][3][idx_val[k]]
        pred_tp[(nama, k)] = tebak
        per_fold_tp.append({"model": nama, "n_konteks": n_konteks, "n_estimators": n_est, "fold": k + 1,
                            "RMSE": rmse(yv_s, tebak), "MAE": mae(yv_s, tebak)})
    print(f"{nama}: selesai dalam {time.time() - t0:.0f} detik")
pf_tp = pd.DataFrame(per_fold_tp)
pf_tp.round(4).to_csv(os.path.join(TABEL, "t5_tabel_tuning_tabpfn_per_fold.csv"), index=False)

baris_tp = []
for n_konteks, n_est in konfig_tp:
    nama = f"TabPFN (konteks={n_konteks}, est={n_est})"
    g = pf_tp[pf_tp["model"] == nama].sort_values("fold")
    baris_tp.append({"Model": nama, "n_konteks": n_konteks, "n_estimators": n_est,
                     "RMSE rata-rata": g["RMSE"].mean(), "RMSE sd": g["RMSE"].std(ddof=1),
                     "MAE rata-rata": g["MAE"].mean(), "MAE sd": g["MAE"].std(ddof=1)})
ringkas_tp = pd.DataFrame(baris_tp).set_index("Model")
ringkas_tp.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning_tabpfn.csv"))
print("\nTotal konfigurasi:", len(ringkas_tp))
print(ringkas_tp.sort_values("RMSE rata-rata").round(2).to_string())

# Aturan pemilihan (ditulis sebelum hasil dilihat, lihat markdown 05.8)
pilih_tp = ringkas_tp["RMSE rata-rata"].idxmin()
kont_pilih, est_pilih = int(ringkas_tp.loc[pilih_tp, "n_konteks"]), int(ringkas_tp.loc[pilih_tp, "n_estimators"])
tepi_kont = kont_pilih in (min(GRID_TABPFN["n_konteks"]), max(GRID_TABPFN["n_konteks"]))
tepi_est = est_pilih in (min(GRID_TABPFN["n_estimators"]), max(GRID_TABPFN["n_estimators"]))
print(f"\nKandidat terpilih : {pilih_tp} | RMSE {ringkas_tp.loc[pilih_tp, 'RMSE rata-rata']:.2f} ± {ringkas_tp.loc[pilih_tp, 'RMSE sd']:.2f}, "
      f"MAE {ringkas_tp.loc[pilih_tp, 'MAE rata-rata']:.2f} ± {ringkas_tp.loc[pilih_tp, 'MAE sd']:.2f}")
print(f"n_konteks di tepi grid: {tepi_kont} | n_estimators di tepi grid: {tepi_est}")
print("Ini kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")

# Grafik: RMSE rata-rata terhadap n_konteks, satu garis per n_estimators
fig, ax = plt.subplots(figsize=(9, 3.8))
for e, warna in zip(GRID_TABPFN["n_estimators"], (BIRU, ORANYE)):
    sub = ringkas_tp[ringkas_tp["n_estimators"] == e].sort_values("n_konteks")
    ax.plot(sub["n_konteks"], sub["RMSE rata-rata"], marker="o", color=warna, label=f"TabPFN, n_estimators={e}")
ax.set_xscale("log"); ax.set_xticks(GRID_TABPFN["n_konteks"]); ax.set_xticklabels(GRID_TABPFN["n_konteks"])
judul(ax, "Tuning TabPFN: RMSE validasi silang", "Rata-rata 5 fold (9 stasiun latih, data uji tidak dibuka). Sumbu vertikal tidak dimulai dari nol")
ax.set(xlabel="n_konteks (skala log)", ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8)
simpan_gambar("t5_06_tuning_tabpfn.png")

### 05.9 TabPFN dibandingkan dengan Ridge, *random forest*, dan *naive forecast*
Kandidat terpilih aturan 05.8, pada *fold* dan baris yang sama dengan tiga model lain. Selisih berpasangan dihitung per *fold* (RMSE TabPFN dikurangi RMSE pembanding, negatif berarti TabPFN lebih baik). Sel ini memeriksa dengan `assert` bahwa skor *naive*, Ridge, dan *random forest* yang dihitung ulang di sini sama dengan 05.2 dan 05.4 **bila** `PREDIK_MAKS_BARIS = None`, sebagai bukti baris yang dinilai sama.

**Aturan membaca hasil** ([Usulan], ditulis sebelum hasil dilihat): TabPFN disebut *dapat dibedakan* dari pembanding hanya bila (a) lebih baik di kelima *fold*, dan (b) selisih rata-rata RMSE lebih besar dari rentang RMSE antar *seed* di 05.10. Selain itu hasilnya dilaporkan sebagai *tidak terbedakan*, apa pun arah selisih rata-ratanya. Dengan lima *fold*, tidak ada klaim "nyata lebih baik" dari selisih rata-rata.

In [ ]:
# Prediksi pembanding pada baris validasi yang sama (subset idx_val; seluruh baris bila PREDIK_MAKS_BARIS = None)
alpha_tp = float(ringkas7.loc[pilih7, "alpha"])
p_acuan = {"Naive forecast": [], model_utama: [], pilih_rf: []}
y_tp = []
for k, ((jam_a, jam_b), (Xa, ya, Xv, yv)) in enumerate(zip(fold_cv, fold7)):
    b = X7.index.isin(jam_b)
    ada = X7[b]["PM2.5"].notna().to_numpy()
    s = idx_val[k]
    y_tp.append(yv[s])
    p_acuan["Naive forecast"].append(X7[b][ada]["PM2.5"].to_numpy()[s])
    p_acuan[model_utama].append(Ridge(alpha=alpha_tp).fit(Xa, ya).predict(Xv)[s])
    p_acuan[pilih_rf].append(pred_rf[(pilih_rf, k)][s])
p_acuan["TabPFN"] = [pred_tp[(pilih_tp, k)] for k in range(5)]
nama_tp = pilih_tp

r_tp = {n: np.array([rmse(y_tp[k], p_acuan[n][k]) for k in range(5)]) for n in p_acuan}
m_tp = {n: np.array([mae(y_tp[k], p_acuan[n][k]) for k in range(5)]) for n in p_acuan}
if PREDIK_MAKS_BARIS is None:          # bukti baris sama: skor ulang harus identik dengan hasil 05.2 dan 05.4
    assert np.allclose(r_tp["Naive forecast"], rmse_naive, atol=1e-9)
    assert np.allclose(r_tp[model_utama], r_ridge, atol=1e-6)
    assert np.allclose(r_tp[pilih_rf], r_rf, atol=1e-6)
    print("Pemeriksaan: skor naive, Ridge, dan random forest identik dengan 05.2 dan 05.4 (baris validasi sama).")
else:
    print(f"Subset {PREDIK_MAKS_BARIS} baris validasi per fold. Skor model lain dihitung ulang pada baris ini, jadi tidak sama dengan 05.2 dan 05.4.")

def baris_tp_model(nama, tampil):
    r, m = r_tp[nama], m_tp[nama]
    return {"Model": tampil, "RMSE rata-rata": r.mean(), "RMSE sd": r.std(ddof=1), "MAE rata-rata": m.mean(), "MAE sd": m.std(ddof=1),
            "Perbaikan RMSE vs naive (%)": 100 * (r_tp["Naive forecast"].mean() - r.mean()) / r_tp["Naive forecast"].mean(),
            "Fold lebih baik dari naive": "n/a" if nama == "Naive forecast" else f"{int((r < r_tp['Naive forecast']).sum())} dari 5"}
tabel_empat = pd.DataFrame([baris_tp_model("Naive forecast", "Naive forecast"), baris_tp_model(model_utama, model_utama),
                            baris_tp_model(pilih_rf, pilih_rf), baris_tp_model("TabPFN", nama_tp)]).set_index("Model")
tabel_empat.round(2).to_csv(os.path.join(TABEL, "t5_tabel_tabpfn_vs_lain.csv"))
d_ridge = r_tp["TabPFN"] - r_tp[model_utama]
d_rfk = r_tp["TabPFN"] - r_tp[pilih_rf]
pd.DataFrame({n: r_tp[n] for n in r_tp} | {"TabPFN minus Ridge": d_ridge, "TabPFN minus RF": d_rfk},
             index=[f"Fold {i}" for i in range(1, 6)]).round(2).to_csv(os.path.join(TABEL, "t5_tabel_tabpfn_vs_lain_per_fold.csv"))

print(tabel_empat.round(2).to_string())
print(f"\nRMSE per fold: naive {np.round(r_tp['Naive forecast'], 1)} | Ridge {np.round(r_tp[model_utama], 1)} | "
      f"RF {np.round(r_tp[pilih_rf], 1)} | TabPFN {np.round(r_tp['TabPFN'], 1)}")
for nama, d in (("Ridge", d_ridge), ("Random forest", d_rfk)):
    print(f"Selisih TabPFN minus {nama}: per fold {np.round(d, 2)} | rata-rata {d.mean():+.2f} ± {d.std(ddof=1):.2f} (sd selisih) | "
          f"TabPFN lebih baik di {int((d < 0).sum())} dari 5 fold")
print(f"Selisih MAE rata-rata (TabPFN minus Ridge): {(m_tp['TabPFN'] - m_tp[model_utama]).mean():+.2f} | "
      f"(TabPFN minus RF): {(m_tp['TabPFN'] - m_tp[pilih_rf]).mean():+.2f}")

fig, ax = plt.subplots(figsize=(9, 4.4))
lebar = 0.2
for j, (n, tampil, warna) in enumerate([("Naive forecast", "Naive forecast", ABU), (model_utama, model_utama, BIRU),
                                         (pilih_rf, pilih_rf, ORANYE), ("TabPFN", nama_tp, "#3b8f5a")]):
    ax.bar(np.arange(5) + (j - 1.5) * lebar, r_tp[n], lebar, color=warna, label=tampil)
ax.set_xticks(range(5)); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
judul(ax, "RMSE validasi silang: naive, Ridge, random forest, dan TabPFN", "Fold dan baris validasi sama untuk keempatnya (9 stasiun latih, data uji tidak dibuka)")
ax.set(ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
simpan_gambar("t5_07_tabpfn_vs_lain_per_fold.png")
tabel_empat.round(2)

### 05.10 Kepekaan TabPFN terhadap *seed*
TabPFN memakai *seed* untuk pemilihan konteks dan untuk pengacakan fitur antar anggota *ensemble*, jadi satu *seed* belum cukup untuk menilai selisih dengan pembanding. Konfigurasi terpilih di 05.8 dilatih ulang dengan tiga *seed* lain (43, 44, 45), tanpa mengubah konfigurasi. Ini pemeriksaan sesudah pemilihan, bukan *tuning* tambahan. Rentang antar *seed* dipakai di aturan membaca 05.9.

In [ ]:
seed_tp = [SEED, 43, 44, 45]
baris_seed_tp, r_seed_tp = [], {}
for sd in seed_tp:
    if sd == SEED:
        r_s = r_tp["TabPFN"]
    else:
        r_s = np.array([rmse(y_tp[k], prediksi_tabpfn(k, kont_pilih, est_pilih, sd)) for k in range(5)])
    r_seed_tp[sd] = r_s
    baris_seed_tp.append({"seed": sd, "RMSE rata-rata": r_s.mean(),
                          "Selisih vs Ridge": (r_s - r_tp[model_utama]).mean(),
                          "Selisih vs RF": (r_s - r_tp[pilih_rf]).mean(),
                          "Fold lebih baik dari Ridge": f"{int((r_s < r_tp[model_utama]).sum())} dari 5",
                          "Fold lebih baik dari RF": f"{int((r_s < r_tp[pilih_rf]).sum())} dari 5"})
tabel_seed_tp = pd.DataFrame(baris_seed_tp).set_index("seed")
tabel_seed_tp.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tabpfn_seed.csv"))
print(tabel_seed_tp.round(2).to_string())
rentang_seed_tp = tabel_seed_tp["RMSE rata-rata"].max() - tabel_seed_tp["RMSE rata-rata"].min()
print(f"\nRentang RMSE rata-rata antar seed: {rentang_seed_tp:.2f} | RMSE Ridge: {r_tp[model_utama].mean():.2f} | RMSE RF: {r_tp[pilih_rf].mean():.2f}")

# Aturan membaca 05.9, diterapkan mekanis (ditulis sebelum hasil dilihat)
for nama_p, d in (("Ridge", d_ridge), ("Random forest", d_rfk)):
    lima_fold = bool((d < 0).all())
    lebih_dari_seed = bool(-d.mean() > rentang_seed_tp)
    putusan = "DAPAT DIBEDAKAN (lebih baik)" if (lima_fold and lebih_dari_seed) else "TIDAK TERBEDAKAN"
    print(f"TabPFN vs {nama_p}: lebih baik di 5 fold = {lima_fold}; selisih rata-rata ({-d.mean():+.2f} perbaikan) > rentang seed = {lebih_dari_seed} -> {putusan}")

### 05.11 Galat TabPFN menurut rentang konsentrasi (diagnosis awal, data validasi)
Pertanyaan yang sama dengan 05.6: apakah TabPFN salah di tempat yang sama dengan Ridge dan *random forest*, terutama pada episode polusi tinggi (di atas 250 µg/m³)? Rentang dan cara membaca bias sama dengan 05.6 (**negatif berarti prediksi terlalu rendah**, dan efek regresi ke rata-rata berlaku untuk model mana pun, jadi yang dibandingkan adalah selisih antar model pada rentang yang sama). Ini analisis pada data validasi, bukan data uji.

In [ ]:
y_g = np.concatenate(y_tp)
pg = {n: np.concatenate(p_acuan[n]) for n in p_acuan}
rentang_tp = pd.cut(pd.Series(y_g), bins=batas_rentang, labels=nama_rentang, right=True, include_lowest=True)
baris_rt = []
for nama in nama_rentang:
    s = (rentang_tp == nama).to_numpy()
    baris = {"Rentang PM2.5 sebenarnya": nama, "Jumlah baris": int(s.sum()), "Rata-rata sebenarnya": y_g[s].mean()}
    for n, kode in (("Naive forecast", "naive"), (model_utama, "Ridge"), (pilih_rf, "RF"), ("TabPFN", "TabPFN")):
        baris[f"Bias {kode}"] = (pg[n][s] - y_g[s]).mean()
    for n, kode in (("Naive forecast", "naive"), (model_utama, "Ridge"), (pilih_rf, "RF"), ("TabPFN", "TabPFN")):
        baris[f"RMSE {kode}"] = rmse(y_g[s], pg[n][s])
    baris_rt.append(baris)
tabel_rentang_tp = pd.DataFrame(baris_rt).set_index("Rentang PM2.5 sebenarnya")
tabel_rentang_tp.round(2).to_csv(os.path.join(TABEL, "t5_tabel_galat_rentang_tabpfn.csv"))
print("RMSE gabungan seluruh fold: " + " | ".join(f"{k} {rmse(y_g, pg[n]):.2f}" for k, n in
      (("naive", "Naive forecast"), ("Ridge", model_utama), ("RF", pilih_rf), ("TabPFN", "TabPFN"))))
print(tabel_rentang_tp.round(1).to_string())
tabpfn_ok = True      # penanda untuk sel Ringkasan: bagian 05.8 sampai 05.11 sudah dijalankan
tabel_rentang_tp.round(1)

**Pembacaan hasil 05.8 sampai 05.11** diisi setelah sel di atas dijalankan, dan hanya dari angka keluaran sel itu. Belum ada hasil TabPFN pada saat bagian ini ditulis (25 Nov 2026): kodenya belum pernah dijalankan dengan bobot model asli. Hal yang harus dilaporkan apa adanya, apa pun hasilnya:

* Skor TabPFN terpilih, selisih berpasangan dengan Ridge dan *random forest*, jumlah *fold* yang lebih baik, dan putusan mekanis dari aturan 05.9 (cetakan terakhir sel 05.10).
* Apakah `n_konteks` atau `n_estimators` terpilih ada di tepi *grid* (keterbatasan, *grid* tidak diperluas).
* Apakah TabPFN memperbaiki *underestimation* di rentang di atas 250 µg/m³, atau salah di tempat yang sama dengan model lain (05.11).
* Waktu jalan dan perangkat (GPU atau CPU), karena biaya ikut menentukan apakah TabPFN layak dibanding Ridge.
* Catatan 05.8 nomor 1 dan 3: konteks TabPFN hanya subsampel data latih, dan kemungkinan kebocoran dari data pelatihan awal bobot belum diverifikasi.
* **Yang tidak boleh dilakukan:** memilih model final berdasarkan selisih yang tidak melewati aturan 05.9, atau menyetel ulang TabPFN setelah melihat skor uji.

## Ringkasan
Angka di bawah dihitung langsung dari data latih di atas.

In [ ]:
n_uji = fitur_uji["kelompok_uji"].value_counts()
print(f"1. Data latih sampai {BATAS - pd.Timedelta(hours=1)}, data uji mulai {BATAS} (tidak dibuka), batas sama untuk 12 stasiun.")
print(f"2. Stasiun latih ({len(STASIUN_LATIH)}): {', '.join(STASIUN_LATIH)}.")
print(f"   Stasiun uji ({len(STASIUN_UJI)}, ditetapkan penulis, sama dengan aturan urutan jarak distribusi {URUTAN_UJI}): {', '.join(STASIUN_UJI)}.")
print(f"3. PM2.5 data latih (9 stasiun): median {pm.median():.0f}, rata-rata {pm.mean():.0f}, miring ke kanan "
      f"(skewness {pm.skew():.2f}). Galat besar saat episode tinggi akan mendominasi RMSE.")
print(f"4. Nilai hilang PM2.5 {latih['PM2.5'].isna().mean():.1%}; {(rentang_hilang <= 3).mean():.0%} rentang nilai hilang "
      f"<= 3 jam, terpanjang {rentang_hilang.max()} jam. Tidak diisi (FFILL_MAKS = {FFILL_MAKS}), diurus imputer di pipeline.")
print(f"5. Pola: tertinggi bulan {nama_bulan[per_bulan.idxmax()-1]}, jam {per_jam.idxmax()}:00.")
print(f"6. Korelasi PM2.5 jam t dengan jam t+24: {korelasi['PM2.5']:.2f}. "
      f"Fitur terkuat lain: {korelasi.drop('PM2.5').abs().idxmax()} ({korelasi.drop('PM2.5')[korelasi.drop('PM2.5').abs().idxmax()]:.2f}).")
print(f"7. {len(KOLOM_FITUR)} fitur siap. Baris latih {len(fitur_latih):,}; baris uji {len(fitur_uji):,} "
      f"({n_uji['stasiun_latih']:,} stasiun latih, {n_uji['stasiun_uji']:,} stasiun uji).")
print(f"8. Baseline naive forecast (CV 5 fold, 9 stasiun, RMSE rata-rata ± sd): "
      f"{tabel_baseline.loc['Naive forecast', 'RMSE rata-rata']:.1f} ± {tabel_baseline.loc['Naive forecast', 'RMSE sd']:.1f}.")
print(f"9. Model lanjutan Ridge (tuning 05.1, 5 nilai alpha, CV 5 fold, RMSE rata-rata ± sd): "
      f"{model_utama} {hasil_ridge.loc[model_utama, 'RMSE rata-rata']:.1f} ± {hasil_ridge.loc[model_utama, 'RMSE sd']:.1f}.")
print(f"   Kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")
print(f"10. Pembanding random forest (tuning 05.3, 6 konfigurasi, CV 5 fold, RMSE rata-rata ± sd): "
      f"{pilih_rf} {r_rf.mean():.1f} ± {r_rf.std(ddof=1):.1f}; selisih dengan Ridge {d_rf.mean():+.2f} (RF lebih baik di {int((d_rf < 0).sum())} dari 5 fold).")
if globals().get("tabpfn_ok"):
    print(f"11. Model tambahan TabPFN (tuning 05.8, 4 konfigurasi, CV 5 fold, RMSE rata-rata ± sd): "
          f"{pilih_tp} {r_tp['TabPFN'].mean():.1f} ± {r_tp['TabPFN'].std(ddof=1):.1f}; selisih dengan Ridge {d_ridge.mean():+.2f} "
          f"(TabPFN lebih baik di {int((d_ridge < 0).sum())} dari 5 fold), dengan RF {d_rfk.mean():+.2f} ({int((d_rfk < 0).sum())} dari 5 fold).")
else:
    print("11. TabPFN (05.8 sampai 05.11) belum dijalankan.")


**Yang perlu diputuskan atau dicek berikutnya**
* Tiga stasiun uji (Dingling, Wanliu, Shunyi) sudah ditetapkan penulis sebelum data uji dibuka. Mengubahnya berarti mengubah keputusan, dan *notebook* dijalankan ulang dari awal.
* Desain 9 stasiun latih dan 3 stasiun uji belum dikonfirmasi dosen.
* Bagian 05 membandingkan Ridge dengan *baseline* pada *fold* yang sama (hanya *cross-validation*). Model pembanding *random forest* sudah ditetapkan dan di-*tuning* (05.3 sampai 05.6). Yang belum: pembekuan konfigurasi model final sebelum data uji dibuka. TabPFN (05.8 sampai 05.11) ditambahkan 25 Nov 2026 sebagai model tambahan dan belum dijalankan pada saat ditulis; hasilnya baru berlaku setelah sel dijalankan dan dibaca dengan aturan 05.9.
* Nilai ekstrem dipertahankan (keputusan 14 Nov 2026). Hasil 02.5 dipakai untuk mengecek apakah keputusan itu masih wajar.
* Berkas uji (`fitur_uji.csv`) dibuka satu kali di tahap 6 lewat `buka_data_uji`, dan skor dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji.